In [17]:
import json
import datetime
import os

# Ruta al archivo de registro (usando ruta relativa correcta)
ruta_registro = os.path.join("r", "videos_procesados.json")

# Cargar el registro
def cargar_registro():
    if not os.path.exists(ruta_registro):
        return {"videos_procesados": {}, "ultima_actualizacion": datetime.datetime.now().isoformat()}
    try:
        with open(ruta_registro, 'r') as f:
            return json.load(f)
    except Exception as e:
        print(f"Error al cargar el registro: {str(e)}")
        return {"videos_procesados": {}, "ultima_actualizacion": datetime.datetime.now().isoformat()}

# Guardar el registro
def guardar_registro(registro):
    try:
        # Crear carpeta si no existe
        os.makedirs(os.path.dirname(ruta_registro), exist_ok=True)
        with open(ruta_registro, 'w') as f:
            json.dump(registro, f, indent=2)
    except Exception as e:
        print(f"Error al guardar el registro: {str(e)}")

# Cargar el registro actual
registro = cargar_registro()

# Eliminar todos los videos procesados
registro["videos_procesados"] = {}

# Actualizar la fecha de la última actualización
registro["ultima_actualizacion"] = datetime.datetime.now().isoformat()

# Guardar el registro actualizado
guardar_registro(registro)

print("Todos los videos han sido eliminados del registro de videos procesados.")



Todos los videos han sido eliminados del registro de videos procesados.


In [18]:
!git clone https://github.com/RizwanMunawar/yolov8-object-tracking.git

fatal: destination path 'yolov8-object-tracking' already exists and is not an empty directory.


In [19]:
!git clone https://github.com/axelqc/YOLO

fatal: destination path 'YOLO' already exists and is not an empty directory.


In [20]:
%pip install -r yolov8-object-tracking//requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [21]:
# Run this before running your script
!set TORCH_WEIGHTS_ONLY=0  # Windows
# or
!export TORCH_WEIGHTS_ONLY=0  # Linux/Mac

In [22]:
%pip install ultralytics==8.0.0
#!pip uninstall torch torchvision -y
#!pip install torch==2.5.0 torchvision==0.16.0
#!pip install torch==2.6.0 torchvision==0.15.0

Note: you may need to restart the kernel to use updated packages.


In [23]:
from pathlib import Path

Path("YOLO/detect_follow.py").rename("yolov8-object-tracking/yolo/v8/detect/detect_follow.py")

FileNotFoundError: [Errno 2] No such file or directory: 'YOLO/detect_follow.py' -> 'yolov8-object-tracking/yolo/v8/detect/detect_follow.py'

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.cluster import KMeans
import os

# Print versions to confirm
print(f"NumPy version: {np.__version__}")
print(f"OpenCV version: {cv2.__version__}")
print(f"Matplotlib version: {plt.matplotlib.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"scikit-learn version: {KMeans.__module__.split('.')[0]}")

print("All imports successful!")

NumPy version: 2.2.6
OpenCV version: 4.11.0
Matplotlib version: 3.10.3
Pandas version: 2.2.3
scikit-learn version: sklearn
All imports successful!


In [ ]:
import os
import cv2
import zipfile
import tempfile
import shutil
import json
import datetime
import subprocess
import re
import math
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from collections import Counter
import pandas as pd
import time
from pathlib import Path

# Global configuration
BASE_DIRECTORY = "/Volumes/Elements/Daniela"  # hay que cambiar esto al path correcto
SHOW_DURATION = False  # false para que no imprima la duracion de los videos
PROCESS_ALL_VIDEOS = True  # vamos a procesar todos los videos
ANALYZE_RESULTS = True  # analizar los resultados de la detección de vehículos inmediatamente después
RESULTS_DIRECTORY = "runs/detect"  # Directorio donde YOLO guarda los resultados

# Función para formatear tamaño de archivos
def formatear_tamaño(bytes, sufijo="B"):
    factor = 1024
    for unidad in ["", "K", "M", "G", "T", "P"]:
        if bytes < factor:
            return f"{bytes:.2f} {unidad}{sufijo}"
        bytes /= factor
    return f"{bytes:.2f} E{sufijo}"

# Función para obtener la duración de un video
def obtener_duracion_video(ruta_video):
    """
    Obtiene la duración de un video en segundos y en formato hora:minuto:segundo
    
    Args:
        ruta_video (str): Ruta al archivo de video
        
    Returns:
        tuple: (duracion_segundos, duracion_formateada)
    """
    try:
        # Abrir el video
        cap = cv2.VideoCapture(ruta_video)
        
        # Verificar si se abrió correctamente
        if not cap.isOpened():
            return None, "No disponible"
        
        # Obtener el número total de frames
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        # Obtener los frames por segundo (FPS)
        fps = cap.get(cv2.CAP_PROP_FPS)
        
        # Calcular la duración en segundos
        duracion_segundos = total_frames / fps if fps > 0 else 0
        
        # Formatear la duración en hora:minuto:segundo
        horas = int(duracion_segundos // 3600)
        minutos = int((duracion_segundos % 3600) // 60)
        segundos = int(duracion_segundos % 60)
        
        duracion_formateada = f"{horas:02d}:{minutos:02d}:{segundos:02d}"
        
        # Liberar el objeto de captura
        cap.release()
        
        return duracion_segundos, duracion_formateada
    except Exception as e:
        print(f"Error al obtener la duración del video: {str(e)}")
        return None, "Error"

# Clase para gestionar el registro de videos procesados
class RegistroVideos:
    def __init__(self, ruta_registro=None):
        # Si no se especifica una ruta, usar el directorio del usuario
        if ruta_registro is None:
            self.ruta_registro = os.path.expanduser("~/videos_procesados.json")
        else:
            self.ruta_registro = ruta_registro
        
        # Inicializar o cargar el registro
        self.registro = self._cargar_registro()
    
    def _cargar_registro(self):
        # Crear el archivo si no existe
        if not os.path.exists(self.ruta_registro):
            # Crear un registro vacío
            registro_inicial = {
                "videos_procesados": {},
                "ultima_actualizacion": datetime.datetime.now().isoformat()
            }
            
            # Guardar el registro inicial
            with open(self.ruta_registro, 'w') as f:
                json.dump(registro_inicial, f, indent=2)
                
            return registro_inicial
        
        # Cargar el registro existente
        try:
            with open(self.ruta_registro, 'r') as f:
                return json.load(f)
        except Exception as e:
            print(f"Error al cargar el registro: {str(e)}")
            return {"videos_procesados": {}, "ultima_actualizacion": datetime.datetime.now().isoformat()}
    
    def esta_procesado(self, ruta_zip, nombre_video):
        # Crear un identificador único para el video
        id_video = f"{ruta_zip}:{nombre_video}"
        
        # Verificar si el video está en el registro
        return id_video in self.registro["videos_procesados"]
    
    def marcar_como_procesado(self, ruta_zip, nombre_video, resultado="completado", excel_path=None):
        # Crear un identificador único para el video
        id_video = f"{ruta_zip}:{nombre_video}"
        
        # Agregar el video al registro
        self.registro["videos_procesados"][id_video] = {
            "ruta_zip": ruta_zip,
            "nombre_video": nombre_video,
            "fecha_procesamiento": datetime.datetime.now().isoformat(),
            "resultado": resultado,
            "excel_path": excel_path
        }
        
        # Actualizar la fecha de última actualización
        self.registro["ultima_actualizacion"] = datetime.datetime.now().isoformat()
        
        # Guardar el registro actualizado
        self._guardar_registro()
    
    def _guardar_registro(self):
        try:
            with open(self.ruta_registro, 'w') as f:
                json.dump(self.registro, f, indent=2)
        except Exception as e:
            print(f"Error al guardar el registro: {str(e)}")
    
    def obtener_estadisticas(self):
        # Obtener estadísticas del registro
        total_videos = len(self.registro["videos_procesados"])
        completados = sum(1 for v in self.registro["videos_procesados"].values() if v["resultado"] == "completado")
        con_errores = sum(1 for v in self.registro["videos_procesados"].values() if v["resultado"] == "error")
        saltados = sum(1 for v in self.registro["videos_procesados"].values() if v["resultado"] == "saltado")
        
        return {
            "total_videos": total_videos,
            "completados": completados,
            "con_errores": con_errores,
            "saltados": saltados,
            "ultima_actualizacion": self.registro["ultima_actualizacion"]
        }
    
    def obtener_videos_procesados_por_zip(self, ruta_zip):
        # Obtener los videos procesados para un ZIP específico
        videos_procesados = []
        
        for id_video, info in self.registro["videos_procesados"].items():
            if info["ruta_zip"] == ruta_zip:
                videos_procesados.append(info["nombre_video"])
        
        return videos_procesados
    
    def obtener_excel_path(self, ruta_zip, nombre_video):
        # Crear un identificador único para el video
        id_video = f"{ruta_zip}:{nombre_video}"
        
        # Verificar si el video está en el registro y tiene una ruta de Excel asociada
        if id_video in self.registro["videos_procesados"]:
            return self.registro["videos_procesados"][id_video].get("excel_path")
        
        return None

# Función para listar videos dentro de un archivo ZIP
def listar_videos_en_zip(ruta_zip, obtener_duracion=False):
    """
    Lista los videos dentro de un archivo ZIP
    
    Args:
        ruta_zip (str): Ruta al archivo ZIP
        obtener_duracion (bool): Si es True, extrae y obtiene la duración de cada video
        
    Returns:
        list: Lista de diccionarios con información de cada video
    """
    videos = []
    extensiones_video = ['.mp4', '.avi', '.mov', '.mkv', '.wmv', '.flv']
    
    try:
        with zipfile.ZipFile(ruta_zip, 'r') as zip_ref:
            for archivo in zip_ref.namelist():
                if any(archivo.lower().endswith(ext) for ext in extensiones_video):
                    info_video = {
                        "nombre": archivo,
                        "tamaño": formatear_tamaño(zip_ref.getinfo(archivo).file_size),
                        "duracion_segundos": None,
                        "duracion": "No extraído"
                    }
                    
                    # Si se requiere la duración, extraer el video temporalmente
                    if obtener_duracion:
                        temp_dir, ruta_temp = extraer_video_temporal(ruta_zip, archivo)
                        
                        if temp_dir and ruta_temp:
                            try:
                                duracion_segundos, duracion = obtener_duracion_video(ruta_temp)
                                info_video["duracion_segundos"] = duracion_segundos
                                info_video["duracion"] = duracion
                            finally:
                                # Limpiar archivos temporales
                                try:
                                    shutil.rmtree(temp_dir)
                                except Exception as e:
                                    print(f"Error al eliminar archivos temporales: {str(e)}")
                    
                    videos.append(info_video)
    except Exception as e:
        print(f"Error al leer {ruta_zip}: {str(e)}")
    
    return videos

# Función para encontrar archivos ZIP
def encontrar_archivos_zip(directorio_base):
    archivos_zip = []
    
    for carpeta_actual, _, archivos in os.walk(directorio_base):
        for archivo in archivos:
            if archivo.lower().endswith('.zip'):
                ruta_completa = os.path.join(carpeta_actual, archivo)
                archivos_zip.append(ruta_completa)
    
    return archivos_zip

# Función para extraer un solo video del ZIP a un archivo temporal
def extraer_video_temporal(ruta_zip, nombre_video):
    try:
        # Crear un directorio temporal
        temp_dir = tempfile.mkdtemp()
        
        # Extraer solo el archivo de video específico
        with zipfile.ZipFile(ruta_zip, 'r') as zip_ref:
            zip_ref.extract(nombre_video, temp_dir)
        
        # Ruta completa al archivo extraído
        ruta_temp = os.path.join(temp_dir, nombre_video)
        
        return temp_dir, ruta_temp
    except Exception as e:
        print(f"Error al extraer {nombre_video}: {str(e)}")
        return None, None

# Función para encontrar el archivo Excel de resultados más reciente
# Modify the encontrar_excel_resultado function to search in all possible directories
def encontrar_excel_resultado(nombre_video, results_dir=None):
    """
    Busca el archivo Excel de resultados más reciente para un video procesado.
    Busca en múltiples directorios posibles donde YOLO podría guardar los resultados.
    """
    import os
    import datetime
    import time
    
    # Obtener nombre base del video (sin extensión y sin ruta)
    nombre_base = os.path.splitext(os.path.basename(nombre_video))[0]
    print(f"Buscando archivos Excel para video con nombre base: {nombre_base}")
    
    # Lista de directorios a buscar
    directorios_busqueda = [
        RESULTS_DIRECTORY,              # El directorio configurado en las variables globales
        "runs/detect",                  # Directorio predeterminado de YOLO
        "runs/detect/train3",           # El directorio que aparece en los mensajes de salida
        os.path.join(os.getcwd(), "runs/detect"),  # Ruta absoluta
        os.path.join(os.getcwd(), "runs/detect/train3"),
        os.path.join(os.getcwd(), "resultados"),   # Directorio donde el script YOLO guarda resultados
        os.path.expanduser("~/resultados"),        # Por si está guardando en el directorio del usuario
    ]
    
    # Agregar directorio específico si se proporciona
    if results_dir:
        directorios_busqueda.insert(0, results_dir)  # Prioridad más alta
    
    print(f"Buscando en los siguientes directorios: {directorios_busqueda}")
    
    # Lista para almacenar todos los Excel encontrados
    excel_encontrados = []
    
    # Buscar en cada directorio
    for directorio in directorios_busqueda:
        if not os.path.exists(directorio):
            print(f"Directorio no encontrado: {directorio}")
            continue
            
        print(f"Buscando en directorio: {directorio}")
        
        # Buscar archivos Excel en este directorio
        for carpeta_actual, _, archivos in os.walk(directorio):
            for archivo in archivos:
                if archivo.endswith('.xlsx'):
                    ruta_completa = os.path.join(carpeta_actual, archivo)
                    
                    # Verificar si el nombre base está en el nombre del archivo
                    if nombre_base in archivo:
                        print(f"¡Encontrado archivo Excel con coincidencia exacta!: {ruta_completa}")
                        tiempo_modificacion = os.path.getmtime(ruta_completa)
                        excel_encontrados.append((ruta_completa, tiempo_modificacion, 1))  # Prioridad 1 (alta)
                    else:
                        # Archivos Excel sin coincidencia exacta tienen menor prioridad
                        tiempo_modificacion = os.path.getmtime(ruta_completa)
                        excel_encontrados.append((ruta_completa, tiempo_modificacion, 2))  # Prioridad 2 (baja)
    
    # Busca Excel modificados en los últimos 5 minutos (para capturar archivos recién creados)
    tiempo_actual = time.time()
    for directorio in directorios_busqueda:
        if not os.path.exists(directorio):
            continue
            
        for carpeta_actual, _, archivos in os.walk(directorio):
            for archivo in archivos:
                if archivo.endswith('.xlsx'):
                    ruta_completa = os.path.join(carpeta_actual, archivo)
                    tiempo_modificacion = os.path.getmtime(ruta_completa)
                    
                    # Si fue modificado en los últimos 5 minutos, considerarlo de alta prioridad
                    if tiempo_actual - tiempo_modificacion < 300:  # 300 segundos = 5 minutos
                        # Verificar si ya está en la lista
                        if not any(ruta_completa == item[0] for item in excel_encontrados):
                            print(f"¡Encontrado archivo Excel reciente!: {ruta_completa}")
                            excel_encontrados.append((ruta_completa, tiempo_modificacion, 0))  # Prioridad 0 (más alta)
    
    # Ordenar por prioridad (primero) y tiempo de modificación (segundo criterio)
    excel_encontrados.sort(key=lambda x: (x[2], -x[1]))
    
    # Imprimir todos los archivos encontrados
    if excel_encontrados:
        print(f"Archivos Excel encontrados (ordenados por prioridad y más recientes):")
        for idx, (path, time, priority) in enumerate(excel_encontrados[:5]):  # Mostrar los 5 más relevantes
            print(f"  {idx+1}. {path} (modificado: {datetime.datetime.fromtimestamp(time)}, prioridad: {priority})")
        
        # Devolver el más prioritario
        return excel_encontrados[0][0]
    
    print("No se encontró ningún archivo Excel relacionado con el video")
    return None

# ===== FUNCIONES PARA ANÁLISIS DE TRAYECTORIAS =====

def determine_optimal_clusters(angle_data, min_clusters=2, max_clusters=6):
    """
    Determine optimal number of clusters using silhouette score.
    This helps find the natural groupings in the data.
    """
    if len(angle_data) < max_clusters:
        return min(len(angle_data), min_clusters)
    
    # Convert angles to points on a unit circle for proper clustering
    x = np.cos(np.radians(angle_data))
    y = np.sin(np.radians(angle_data))
    
    # Combine into feature array
    features = np.column_stack((x, y))
    
    best_score = -1
    best_n_clusters = min_clusters
    
    # Try different cluster counts and find the best one
    for n_clusters in range(min_clusters, min(max_clusters + 1, len(angle_data))):
        kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        cluster_labels = kmeans.fit_predict(features)
        
        # Calculate silhouette score
        try:
            score = silhouette_score(features, cluster_labels)
            print(f"  Testing {n_clusters} clusters: silhouette score = {score:.4f}")
            
            if score > best_score:
                best_score = score
                best_n_clusters = n_clusters
        except:
            # If silhouette score fails (e.g., only one sample in cluster)
            continue
    
    print(f"  Optimal number of clusters: {best_n_clusters} (score: {best_score:.4f})")
    return best_n_clusters

def process_vehicle_data(file_path, sheet_name="Datos_Detallados", min_clusters=2, max_clusters=6, min_displacement=5):
    """Process vehicle data and classify directions with adaptive clustering."""
    print(f"Procesando análisis de trayectorias para: {file_path}...")
    
    # Try to extract video ID from filename
    video_id = os.path.splitext(os.path.basename(file_path))[0]
    print(f"Archivo: {video_id}")
    
    # Load data
    try:
        df = pd.read_excel(file_path, sheet_name=sheet_name)
    except Exception as e:
        print(f"Error al cargar archivo Excel: {e}")
        return None, None, None
    
    print(f"Cargados {len(df)} registros de detecciones de vehículos")
    
    # Skip header row if it's duplicated
    if "ID" in df.columns and df.iloc[0]["ID"] == "ID":
        df = df.iloc[1:].reset_index(drop=True)
        
    # Convert ID to numeric if it's not already
    if "ID" in df.columns:
        try:
            df["ID"] = pd.to_numeric(df["ID"])
            # Keep only the last record for each vehicle (final trajectory)
            df.drop_duplicates(subset="ID", keep="last", inplace=True)
            print(f"Después de eliminar duplicados: {len(df)} vehículos únicos")
        except Exception as e:
            print(f"Error al convertir ID a numérico: {e}")
    
    # Extract trajectory features
    try:
        # Find trajectory column
        traj_col = None
        for col in df.columns:
            if 'trayectoria' in col.lower() or 'trajectory' in col.lower():
                traj_col = col
                break
        
        if not traj_col:
            print("Error: No se pudo encontrar la columna de trayectoria")
            return None, None, None
        
        # Parse trajectories
        df['trajectory_points'] = df[traj_col].apply(parse_trajectory)
        
        # Calculate direction features
        vectors = [calculate_trajectory_vector(points) for points in df['trajectory_points']]
        df['dx'] = [v[0] if v is not None and v[0] is not None else 0 for v in vectors]
        df['dy'] = [v[1] if v is not None and v[1] is not None else 0 for v in vectors]
        df['angle'] = [v[2] if v is not None and v[2] is not None else -1 for v in vectors]
        
        # Calculate displacement distance
        df['distance'] = np.sqrt(df['dx']**2 + df['dy']**2)
        
        # Filter out invalid trajectories (too short or no movement)
        valid_df = df[df['distance'] > min_displacement].copy()
        print(f"Encontradas {len(valid_df)} trayectorias válidas con movimiento suficiente")
        
        if len(valid_df) < min_clusters:
            print(f"Advertencia: No hay suficientes trayectorias válidas para agrupamiento. Se necesitan al menos {min_clusters}.")
            return df, None, None
            
        # Get valid angles for clustering
        valid_angles = valid_df[valid_df['angle'] >= 0]['angle'].values
        
        # Determine optimal number of clusters
        print("Determinando número óptimo de grupos (clusters)...")
        n_clusters = determine_optimal_clusters(valid_angles, min_clusters, max_clusters)
        
        # Perform clustering with optimal number of clusters
        print(f"Agrupando con {n_clusters} direcciones principales...")
        cluster_angles, valid_df = cluster_directions(valid_df, n_clusters)
        
        if cluster_angles is None:
            print("El agrupamiento falló.")
            return df, None, None
        
        # Name clusters
        cluster_to_name, sorted_angles = name_direction_clusters(cluster_angles, video_id)
        
        # Assign direction names
        valid_df = assign_direction_names(valid_df, cluster_to_name)
        
        # Print cluster information
        print("\nGrupos de dirección identificados:")
        direction_counts = Counter(valid_df['direction'])
        for cluster_idx, angle in enumerate(cluster_angles):
            name = cluster_to_name[cluster_idx]
            count = direction_counts.get(name, 0)
            percent = (count / len(valid_df)) * 100 if len(valid_df) > 0 else 0
            print(f"  Grupo {cluster_idx}: {name} (ángulo: {angle:.1f}°, conteo: {count}, {percent:.1f}%)")
        
        # Check for potential overclusterng
        if n_clusters > 3:
            # Look for similar angles that might be the same direction
            for i in range(n_clusters):
                for j in range(i+1, n_clusters):
                    # Check if angles are within 30 degrees of each other
                    angle_diff = min(abs(cluster_angles[i] - cluster_angles[j]), 
                                    360 - abs(cluster_angles[i] - cluster_angles[j]))
                    if angle_diff < 30:
                        print(f"  Nota: Los grupos {i} y {j} tienen ángulos similares " 
                              f"({cluster_angles[i]:.1f}° y {cluster_angles[j]:.1f}°)")
                        print(f"  Estos podrían representar la misma dirección general")
        
        return valid_df, cluster_angles, cluster_to_name
        
    except Exception as e:
        print(f"Error en el análisis de trayectoria: {e}")
        import traceback
        traceback.print_exc()
        return df, None, None

def parse_trajectory(trajectory_str):
    """Parse trajectory string into list of coordinate tuples."""
    try:
        # Check if input is a string
        if not isinstance(trajectory_str, str):
            return []
            
        # Extract all (x,y) coordinates from the trajectory string
        coordinates = re.findall(r'\((\d+),(\d+)\)', trajectory_str)
        
        # Convert to list of (x, y) tuples with integer values
        points = [(int(x), int(y)) for x, y in coordinates]
        
        return points
    except Exception as e:
        print(f"Error parsing trajectory: {e} for input: {trajectory_str}")
        return []

def calculate_trajectory_vector(points):
    """Calculate the direction vector of a trajectory."""
    if not points or len(points) < 2:
        return None, None, None
    
    try:
        # Use first and last point for overall direction
        start_x, start_y = points[0]
        end_x, end_y = points[-1]
        
        # Calculate displacement vector
        dx = end_x - start_x
        dy = end_y - start_y
        
        # Calculate vector length (distance traveled)
        distance = math.sqrt(dx**2 + dy**2)
        
        # Calculate angle in degrees (0 = East, going counterclockwise)
        angle = math.degrees(math.atan2(dy, dx))
        
        # Normalize angle to 0-360 range
        if angle < 0:
            angle += 360
            
        return dx, dy, angle
    except Exception as e:
        print(f"Error calculating trajectory vector: {e}")
        return None, None, None
    
def find_optimal_clusters(df, min_clusters=2, max_clusters=5):
    """
    Determine the optimal number of direction clusters based on angle distribution.
    """
    from sklearn.cluster import KMeans
    from sklearn.metrics import silhouette_score
    
    # Extract valid angles
    angles = df['angle'].dropna().values
    
    if len(angles) < max_clusters:
        return min(len(angles), min_clusters)
    
    # Convert angles to points on unit circle for proper clustering
    x = np.cos(np.radians(angles))
    y = np.sin(np.radians(angles))
    
    # Combine into feature array
    features = np.column_stack((x, y))
    
    best_score = -1
    best_n_clusters = min_clusters
    
    # Try different cluster counts and find the best one
    for n_clusters in range(min_clusters, min(max_clusters + 1, len(angles))):
        kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        cluster_labels = kmeans.fit_predict(features)
        
        # Calculate silhouette score if possible
        if len(set(cluster_labels)) > 1:  # Need at least 2 clusters
            try:
                score = silhouette_score(features, cluster_labels)
                print(f"  Testing {n_clusters} clusters: silhouette score = {score:.4f}")
                
                if score > best_score:
                    best_score = score
                    best_n_clusters = n_clusters
            except:
                continue
    
    print(f"  Optimal number of clusters: {best_n_clusters}")
    return best_n_clusters


def cluster_directions(df, n_clusters=4):
    """Cluster vehicle trajectories into dominant directions."""
    from sklearn.cluster import KMeans
    
    # Get valid angles
    valid_angles = df[df['angle'] >= 0]['angle'].values
    
    if len(valid_angles) < n_clusters:
        print(f"Warning: Not enough valid trajectories. Found {len(valid_angles)}, need at least {n_clusters}")
        return None, None
    
    # Convert angles to x,y on unit circle for proper clustering
    x = np.cos(np.radians(valid_angles))
    y = np.sin(np.radians(valid_angles))
    
    # Combine into feature array
    features = np.column_stack((x, y))
    
    # Perform clustering
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    clusters = kmeans.fit_predict(features)
    
    # Get cluster centers
    centers = kmeans.cluster_centers_
    
    # Convert cluster centers back to angles
    cluster_angles = np.degrees(np.arctan2(centers[:, 1], centers[:, 0]))
    cluster_angles = np.mod(cluster_angles, 360)  # Normalize to 0-360
    
    # Function to assign angle to nearest cluster
    def find_nearest_cluster(angle):
        if angle < 0:
            return -1
        
        # Convert angle to point on unit circle
        x = np.cos(np.radians(angle))
        y = np.sin(np.radians(angle))
        
        # Find nearest cluster
        distances = [np.sqrt((x - cx)**2 + (y - cy)**2) for cx, cy in centers]
        return np.argmin(distances)
    
    # Assign each trajectory to a cluster
    df['direction_cluster'] = df['angle'].apply(find_nearest_cluster)
    
    return cluster_angles, df

def name_direction_clusters(cluster_angles, video_id=None):
    """Assign names to direction clusters based on their angles."""
    # Sort clusters by angle
    sorted_indices = np.argsort(cluster_angles)
    sorted_angles = cluster_angles[sorted_indices]
    
    # If video_id is provided, use it as prefix for direction names
    if video_id is not None:
        video_id = video_id.replace("_manual", "")
        prefix = f"{video_id}_" 
    else:
        prefix = ""
    
    # Create direction names based on number of clusters
    direction_names = [f"{prefix}Direction_{i+1}" for i in range(len(sorted_angles))]
    
    # Create mapping from cluster index to direction name
    cluster_to_name = {}
    for i, idx in enumerate(sorted_indices):
        cluster_to_name[idx] = direction_names[i]
    
    return cluster_to_name, sorted_angles

def enhance_vehicle_trajectory_visualization(input_excel=None, video_frame=None, video_id="traffic_video"):
    """
    Create an enhanced visualization of vehicle trajectories with clear direction indicators.
    
    Args:
        input_excel: Path to Excel file with trajectory data (if None, generate sample data)
        video_frame: Path to video frame image to use as background (if None, use blank canvas)
        video_id: Identifier for the video (used in naming)
    
    Returns:
        Path to saved visualization image
    """
    # Set up figure size
    plt.figure(figsize=(12, 9), dpi=150)
    
    # Load or generate data
    if input_excel and os.path.exists(input_excel):
        # Try to load real data
        try:
            df = pd.read_excel(input_excel, sheet_name="Datos_Detallados")
            print(f"Loaded {len(df)} vehicle records from Excel")
        except Exception as e:
            print(f"Error loading Excel: {e}")
            df = None
    else:
        df = None
    
    # If no data available, generate synthetic data
    if df is None:
        print("Creating synthetic vehicle trajectory data...")
        df = generate_synthetic_data(video_id, num_vehicles=20)
    
    # Process trajectory data
    df['trajectory_points'] = df['Trayectoria'].apply(parse_trajectory)
    
    # Calculate trajectory vectors
    vectors = [calculate_trajectory_vector(points) for points in df['trajectory_points']]
    df['dx'] = [v[0] if v is not None and v[0] is not None else 0 for v in vectors]
    df['dy'] = [v[1] if v is not None and v[1] is not None else 0 for v in vectors]
    df['angle'] = [v[2] if v is not None and v[2] is not None else -1 for v in vectors]
    
    # Calculate displacement distance
    df['distance'] = np.sqrt(df['dx']**2 + df['dy']**2)
    
    # Filter out invalid trajectories (too short or no movement)
    valid_df = df[df['distance'] > 5].copy()
    print(f"Found {len(valid_df)} valid trajectories with sufficient movement")
    
    # Determine optimal number of clusters
    n_clusters = find_optimal_clusters(valid_df, min_clusters=2, max_clusters=5)
    
    # Perform clustering
    cluster_angles, valid_df = cluster_directions(valid_df, n_clusters)
    
    # Name clusters
    cluster_to_name, sorted_angles = name_direction_clusters(cluster_angles, video_id)


    def get_direction_name(cluster_idx):
        if cluster_idx < 0:
            return "Unknown"
        return cluster_to_name.get(cluster_idx, f"Direction_{cluster_idx}")
    
    valid_df['direction'] = valid_df['direction_cluster'].apply(get_direction_name)
    
    # Load background image if provided
    if video_frame and os.path.exists(video_frame):
        bg_image = plt.imread(video_frame)
        frame_height, frame_width = bg_image.shape[:2]
        plt.imshow(bg_image, extent=[0, frame_width, frame_height, 0])
    else:
        # Create a clean white background with grid
        frame_width, frame_height = 1280, 720
        plt.gca().set_facecolor('white')
        plt.grid(True, alpha=0.3)
    
    # Set up plot area
    plt.xlim(0, frame_width)
    plt.ylim(frame_height, 0)  # Inverted y-axis to match image coordinates
    
    # Extract trajectory coordinates for scaling
    all_x, all_y = [], []
    for points in valid_df['trajectory_points']:
        if len(points) >= 2:
            x, y = zip(*points)
            all_x.extend(x)
            all_y.extend(y)
    
    # Calculate scaling to fit trajectories to frame
    if all_x and all_y:
        min_x, max_x = min(all_x), max(all_x)
        min_y, max_y = min(all_y), max(all_y)
        
        # Calculate scaling factors with margins
        margin_x = frame_width * 0.1
        margin_y = frame_height * 0.1
        
        # Available space
        avail_width = frame_width - 2 * margin_x
        avail_height = frame_height - 2 * margin_y
        
        # Calculate scaling
        scale_x = avail_width / (max_x - min_x) if max_x > min_x else 1
        scale_y = avail_height / (max_y - min_y) if max_y > min_y else 1
        
        # Use smaller scale to maintain aspect ratio
        scale = min(scale_x, scale_y) * 0.9
        
        # Calculate offset to center
        offset_x = margin_x + (avail_width - scale * (max_x - min_x)) / 2
        offset_y = margin_y + (avail_height - scale * (max_y - min_y)) / 2
        
        print(f"Scaling trajectories by factor {scale:.4f}")
        
        # Scale all trajectories
        for idx, row in valid_df.iterrows():
            points = row['trajectory_points']
            if len(points) >= 2:
                new_points = [(offset_x + x * scale, offset_y + y * scale) for x, y in points]
                valid_df.at[idx, 'scaled_points'] = new_points
    else:
        # If no valid coordinates, just pass through
        for idx, row in valid_df.iterrows():
            valid_df.at[idx, 'scaled_points'] = row['trajectory_points']
    
    # Create color scheme for directions
    colors = plt.cm.tab10(np.linspace(0, 1, len(cluster_to_name)))
    direction_colors = {}
    
    for i, (cluster_idx, direction) in enumerate(cluster_to_name.items()):
        direction_colors[direction] = colors[i]
    
    # Add 'Unknown' direction if present
    if 'Unknown' in valid_df['direction'].values:
        direction_colors['Unknown'] = (0.7, 0.7, 0.7, 1.0)  # Gray
    
    # Calculate center of visualization (for summary arrows)
    center_x = frame_width / 2
    center_y = frame_height / 2
    
    # Draw trajectories
    for _, row in valid_df.iterrows():
        direction = row['direction']
        points = row.get('scaled_points', row['trajectory_points'])
        
        if len(points) >= 2:
            # Get trajectory coordinates
            xs, ys = zip(*points)
            
            # Get color for this direction
            color = direction_colors.get(direction, (0, 0, 0))
            
            # Plot trajectory line
            plt.plot(xs, ys, '-', color=color, linewidth=2, alpha=0.7)
            
            # Mark start point
            plt.plot(xs[0], ys[0], 'o', color=color, markersize=5)
            
            # Add arrow to end point
            arrow_scale = min(frame_width, frame_height) * 0.005
            if len(xs) >= 2:
                # Use direction from second-last to last point for arrow
                plt.arrow(xs[-2], ys[-2], xs[-1] - xs[-2], ys[-1] - ys[-2],
                         head_width=arrow_scale*3, head_length=arrow_scale*4,
                         fc=color, ec=color, alpha=0.8)
    
    # Draw summary arrows from center showing main directions
    direction_counts = Counter(valid_df['direction'])
    arrow_length = min(frame_width, frame_height) * 0.3
    
    # Draw cluster summary arrows
    for cluster_idx, angle_deg in enumerate(cluster_angles):
        direction = cluster_to_name[cluster_idx]
        count = direction_counts.get(direction, 0)
        
        # Skip directions with few vehicles
        if count < 2:
            continue
            
        # Calculate percentage
        percentage = count / len(valid_df) * 100 if len(valid_df) > 0 else 0
        
        # Get color
        color = direction_colors.get(direction, (0, 0, 0))
        
        # Convert angle to radians
        angle_rad = np.radians(angle_deg)
        
        # Calculate arrow endpoint
        end_x = center_x + arrow_length * np.cos(angle_rad)
        end_y = center_y + arrow_length * np.sin(angle_rad)
        
        # Scale thickness based on percentage
        thickness = max(3, min(15, int(percentage / 3)))
        
        # Draw arrow
        plt.arrow(center_x, center_y, 
                 end_x - center_x, end_y - center_y,
                 head_width=thickness*1.5, head_length=thickness*2,
                 fc=color, ec=color, width=thickness/2, alpha=0.8)
        
        # Add direction label
        text_offset = arrow_length * 0.08
        text_x = center_x + (arrow_length + text_offset) * np.cos(angle_rad)
        text_y = center_y + (arrow_length + text_offset) * np.sin(angle_rad)
        
        # Create clean label text
        angle_cardinal = get_cardinal_direction(angle_deg)
        label = f"{direction}\n({angle_cardinal}, {angle_deg:.1f}°)\n{count} vehicles ({percentage:.1f}%)"
        
        # Add a white background to the text for better visibility
        bbox_props = dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.8)
        
        # Add text with background
        plt.text(text_x, text_y, label, 
                ha='center', va='center', 
                fontsize=9, fontweight='bold',
                bbox=bbox_props)
    
    # Add legend
    legend_elements = []
    for direction, color in direction_colors.items():
        element = Line2D([0], [0], color=color, lw=4, label=direction)
        legend_elements.append(element)
    
    # Position legend with transparent background
    legend = plt.legend(handles=legend_elements, loc='upper right', 
                      title="Vehicle Directions", 
                      framealpha=0.8)
    
    # Add title
    plt.title(f"Vehicle Trajectory Analysis - {video_id}", fontsize=16, pad=20)
    
    # Remove axis ticks for cleaner look
    plt.xticks([])
    plt.yticks([])
    plt.box(False)
    
    # Save visualization
    output_path = f"{video_id}_enhanced_trajectories.png"
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"Enhanced visualization saved to: {output_path}")
    return output_path

def get_cardinal_direction(angle_deg):
    """Convert angle in degrees to cardinal direction."""
    # Define direction ranges
    directions = {
        'N': (337.5, 22.5),    # North
        'NE': (22.5, 67.5),    # Northeast
        'E': (67.5, 112.5),    # East
        'SE': (112.5, 157.5),  # Southeast
        'S': (157.5, 202.5),   # South
        'SW': (202.5, 247.5),  # Southwest
        'W': (247.5, 292.5),   # West
        'NW': (292.5, 337.5)   # Northwest
    }
    
    # Normalize angle to 0-360 range
    angle_deg = angle_deg % 360
    
    # Find matching direction
    for direction, (start, end) in directions.items():
        if start > end:  # Handles the North case that wraps around
            if angle_deg >= start or angle_deg < end:
                return direction
        else:
            if start <= angle_deg < end:
                return direction
    
    return 'N'  # Default fallback
            
# Add this function to your analyze_excel_file function
def analyze_excel_file(excel_path, video_path=None):
    """
    Analyzes an Excel file with vehicle detection data.
    If video_path is provided, uses the first frame for trajectory visualization.
    """
    try:
        # Process data with adaptive clustering
        df, cluster_angles, cluster_to_name = process_vehicle_data(
            excel_path, 
            min_clusters=2,
            max_clusters=5
        )
        
        if df is not None:
            # Save results to new sheets in the original Excel file
            with pd.ExcelWriter(excel_path, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
                # Create summary sheet with vehicle direction counts
                if cluster_to_name is not None:
                    # Get direction counts
                    direction_counts = Counter(df['direction'])
                    
                    # Create summary dataframe
                    summary_df = pd.DataFrame({
                        'Direction': list(direction_counts.keys()),
                        'VehicleCount': list(direction_counts.values()),
                    })
                    
                    # Sort by count (descending)
                    summary_df = summary_df.sort_values('VehicleCount', ascending=False)
                    
                    # Add percentages
                    total = summary_df['VehicleCount'].sum()
                    summary_df['Percentage'] = summary_df['VehicleCount'] / total * 100
                    
                    # Save to Excel
                    summary_df.to_excel(writer, sheet_name="Direction_Summary", index=False)
                    
                    # Also save cluster info
                    cluster_info = pd.DataFrame({
                        'Cluster': list(cluster_to_name.keys()),
                        'DirectionName': list(cluster_to_name.values()),
                        'Angle': [cluster_angles[i] for i in cluster_to_name.keys()]
                    })
                    cluster_info.to_excel(writer, sheet_name="Direction_Clusters", index=False)
                
                # Save complete analyzed data
                df.to_excel(writer, sheet_name="Analyzed_Trajectories", index=False)
            try:
                visualize_trajectories_like_example(df, cluster_angles, cluster_to_name, excel_path, video_path)
            except Exception as e:
                print(f"Error with actual trajectory visualization: {e}")
                try:
                    # Fall back to simplified method if the first fails
                    visualize_trajectories_like_example(df, cluster_angles, cluster_to_name, excel_path, video_path)
                except Exception as e:
                    print(f"Error with simplified visualization: {e}")
            
            print("\nAnalysis completed!")
            print(f"Processed {len(df)} valid vehicle trajectories")
            print("\nResults added as new sheets to the original Excel file:")
            print("- Analyzed_Trajectories: Detailed trajectory analysis")
            print("- Direction_Summary: Vehicle count by direction")
            print("- Direction_Clusters: Information about identified direction clusters")
            
            return True
        
    except Exception as e:
        print(f"Error processing file: {e}")
        import traceback
        traceback.print_exc()
        
    return False

def visualize_trajectories_fixed(df, cluster_angles, cluster_to_name, file_path, video_path=None):
    """
    Creates trajectory visualization without using mask operations that cause errors.
    
    Args:
        df: DataFrame with trajectory data
        cluster_angles: Array of angles for each direction cluster
        cluster_to_name: Dictionary mapping cluster indices to direction names
        file_path: Path to Excel file (used for naming output)
        video_path: Path to video file to extract clean frame
    
    Returns:
        Path to saved visualization image
    """
    import cv2
    import numpy as np
    from collections import Counter
    import os
    
    # Create output filename
    base_filename = file_path.rsplit('.', 1)[0]
    video_id = os.path.splitext(os.path.basename(file_path))[0]
    video_id = video_id.replace("_manual", "")
    
    # Get clean frame from video or create blank canvas
    if video_path and os.path.exists(video_path):
        cap = cv2.VideoCapture(video_path)
        ret, frame = cap.read()
        cap.release()
        if not ret:
            frame = np.ones((720, 1280, 3), dtype=np.uint8) * 240
    else:
        frame = np.ones((720, 1280, 3), dtype=np.uint8) * 240
    
    # Get frame dimensions
    frame_height, frame_width = frame.shape[:2]
    
    # Create distinct colors for directions - BGR format for OpenCV
    colors = [
        (255, 0, 0),    # Blue
        (0, 0, 255),    # Red
        (128, 0, 128),  # Purple/Magenta
        (0, 255, 255),  # Yellow
        (0, 255, 0),    # Green
        (255, 0, 255)   # Pink
    ]
    
    # Assign colors to directions
    direction_colors = {}
    for i, (cluster_idx, direction) in enumerate(cluster_to_name.items()):
        direction_colors[direction] = colors[i % len(colors)]
    
    # Count vehicles in each direction
    direction_counts = Counter(df['direction'])
    total_vehicles = sum(direction_counts.values())
    
    # Draw actual trajectory lines by direction
    if 'trajectory_points' in df.columns:
        for _, row in df.iterrows():
            direction = row['direction']
            points = row.get('trajectory_points', [])
            
            if len(points) >= 2:
                # Get color for this direction
                color = direction_colors.get(direction, (200, 200, 200))
                
                # Convert points to integer tuples
                points = [(int(x), int(y)) for x, y in points]
                
                # Draw trajectory line
                for i in range(len(points) - 1):
                    cv2.line(frame, points[i], points[i+1], color, 2, cv2.LINE_AA)
                
                # Add small dot at start
                cv2.circle(frame, points[0], 3, color, -1)
                
                # Add arrow to final segment
                if len(points) >= 2:
                    # Draw arrowhead on final segment
                    cv2.arrowedLine(frame, points[-2], points[-1], color, 2, cv2.LINE_AA, tipLength=0.3)
    
    # Add title
    title = f"Análisis de Trayectorias de Vehículos - {video_id}"
    cv2.putText(frame, title, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 2, cv2.LINE_AA)
    
    # Draw legend directly on frame - simplified approach to avoid mask operations
    # Position in top-right corner
    legend_y_start = 50
    legend_x = frame_width - 400
    
    # Add legend header
    cv2.putText(frame, "Direcciones Identificadas", 
                (legend_x, legend_y_start), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 1, cv2.LINE_AA)
    
    # Draw legend background
    legend_height = 40 + (len(cluster_to_name) * 25)
    legend_width = 400
    cv2.rectangle(frame, 
                 (legend_x - 10, legend_y_start - 30),
                 (legend_x + legend_width - 10, legend_y_start + legend_height - 20),
                 (255, 255, 255), -1)  # White filled rectangle
    cv2.rectangle(frame, 
                 (legend_x - 10, legend_y_start - 30),
                 (legend_x + legend_width - 10, legend_y_start + legend_height - 20),
                 (0, 0, 0), 1)  # Black outline
    
    # Re-add legend header on top of white background
    cv2.putText(frame, "Direcciones Identificadas", 
                (legend_x, legend_y_start), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 1, cv2.LINE_AA)
    
    # Add legend items
    y_offset = legend_y_start + 30
    for cluster_idx, angle_deg in enumerate(cluster_angles):
        direction = cluster_to_name[cluster_idx]
        color = direction_colors[direction]
        
        # Draw line sample
        cv2.line(frame, (legend_x + 10, y_offset), (legend_x + 50, y_offset), color, 2, cv2.LINE_AA)
        cv2.arrowedLine(frame, (legend_x + 30, y_offset), (legend_x + 50, y_offset), color, 2, cv2.LINE_AA)
        
        # Draw label with angle
        label = f"{direction} ({angle_deg:.1f}°)"
        cv2.putText(frame, label, (legend_x + 60, y_offset + 5), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
        
        y_offset += 25
    
    # Draw stats box in bottom-left corner (simplified approach)
    stats_x = 10
    stats_y = frame_height - 160
    
    # Create stats text
    stats_text = []
    stats_text.append("Estadísticas de Direcciones:")
    
    # Sort directions by count (descending)
    sorted_directions = sorted(direction_counts.items(), 
                              key=lambda x: x[1], reverse=True)
    
    for direction, count in sorted_directions:
        percentage = (count / total_vehicles) * 100 if total_vehicles > 0 else 0
        stats_text.append(f"{direction}: {count} ({percentage:.1f}%)")
    
    # Calculate stats box size
    stats_height = 30 + (len(stats_text) * 20)
    stats_width = 350
    
    # Draw stats background
    cv2.rectangle(frame, 
                 (stats_x, stats_y),
                 (stats_x + stats_width, stats_y + stats_height),
                 (255, 255, 255), -1)  # White filled rectangle
    cv2.rectangle(frame, 
                 (stats_x, stats_y),
                 (stats_x + stats_width, stats_y + stats_height),
                 (0, 0, 0), 1)  # Black outline
    
    # Add stats text
    for i, text in enumerate(stats_text):
        y_pos = stats_y + 25 + (i * 20)
        cv2.putText(frame, text, (stats_x + 10, y_pos), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
    
    # Add camera label in bottom right
    cv2.putText(frame, "Camera 01", (frame_width - 120, frame_height - 20),
               cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2, cv2.LINE_AA)
    
    # Save visualization
    output_path = f"{base_filename}_fixed.png"
    cv2.imwrite(output_path, frame)
    
    print(f"Fixed visualization saved to: {output_path}")
    return output_path

def procesar_archivos_zip_automatizado(batch_size=5, max_duration_seconds=300):
    """
    Versión automatizada para procesar archivos ZIP con videos en lotes.
    Extrae y procesa batch_size videos a la vez.
    
    Args:
        batch_size (int): Número de videos a procesar por lote
        max_duration_seconds (int): Duración máxima en segundos para procesar un video
    """
    
    try:
        # Definir el directorio de resultados en la carpeta actual del proyecto
        resultados_dir = os.path.join(os.getcwd(), "resultados")
        os.makedirs(resultados_dir, exist_ok=True)
        print(f"Directorio de resultados: {resultados_dir}")

        print("\n" + "="*50)
        print(f"PROCESAMIENTO AUTOMATIZADO DE VIDEOS EN ARCHIVOS ZIP (EN LOTES DE {batch_size})")
        print(f"Omitiendo videos más largos de {max_duration_seconds} segundos ({max_duration_seconds/60:.1f} minutos)")
        print("="*50)
        
        # Inicializar el registro
        registro = RegistroVideos()
        
        # Mostrar estadísticas
        estadisticas = registro.obtener_estadisticas()
        print(f"Registro existente: {estadisticas['total_videos']} videos procesados, "
              f"{estadisticas['completados']} completados, {estadisticas['con_errores']} con errores")
        
        # Buscar archivos ZIP
        print(f"\nBuscando archivos ZIP en: {BASE_DIRECTORY}")
        archivos_zip = encontrar_archivos_zip(BASE_DIRECTORY)
        
        if not archivos_zip:
            print(f"No se encontraron archivos ZIP en {BASE_DIRECTORY}.")
            return
        
        print(f"Se encontraron {len(archivos_zip)} archivos ZIP.")
        
        # Procesar cada archivo ZIP
        for i, ruta_zip in enumerate(archivos_zip):
            nombre_zip = os.path.basename(ruta_zip)
            print(f"\n{'-'*80}")
            print(f"[{i+1}/{len(archivos_zip)}] Procesando archivo: {nombre_zip}")
            
            # Listar los videos - necesitamos duración, así que True en obtener_duracion
            videos_info = listar_videos_en_zip(ruta_zip, obtener_duracion=True)
            videos_procesados = registro.obtener_videos_procesados_por_zip(ruta_zip)
            
            print(f"Videos encontrados: {len(videos_info)}")
            print(f"Videos previamente procesados: {len(videos_procesados)}")
            
            # Filtrar videos para procesar solo los que sean cortos
            videos_filtrados = []
            for video in videos_info:
                # Comprobar duración
                duracion_segundos = video.get("duracion_segundos")
                
                if duracion_segundos is None:
                    print(f"Advertencia: No se pudo determinar la duración de {video['nombre']}. Se excluirá por precaución.")
                    continue
                    
                if duracion_segundos > max_duration_seconds:
                    print(f"Omitiendo {video['nombre']} - Duración: {video['duracion']} (mayor a {max_duration_seconds/60:.1f} minutos)")
                    # Marcar como saltado en el registro
                    registro.marcar_como_procesado(ruta_zip, video['nombre'], "saltado por duración")
                    continue
                    
                videos_filtrados.append(video)
            
            print(f"Videos que cumplen el criterio de duración (<= {max_duration_seconds/60:.1f} minutos): {len(videos_filtrados)}")
            
            # Procesar videos filtrados en lotes
            for j in range(0, len(videos_filtrados), batch_size):
                # Obtener el lote actual de videos
                batch_videos = videos_filtrados[j:j+batch_size]
                print(f"\nProcesando lote {j//batch_size + 1} ({len(batch_videos)} videos)")
                
                # Lista para mantener seguimiento de los directorios temporales
                temp_dirs = []
                temp_paths = []
                
                try:
                    # Primero extraer todos los videos del lote
                    for k, video_info in enumerate(batch_videos):
                        nombre_video = video_info["nombre"]
                        
                        print(f"\n[{j+k+1}/{len(videos_filtrados)}] Extrayendo: {nombre_video}")
                        print(f"Tamaño: {video_info['tamaño']}")
                        print(f"Duración: {video_info['duracion']} ({video_info['duracion_segundos']:.1f} segundos)")
                        
                        # Verificar si el video ya ha sido procesado
                        if not PROCESS_ALL_VIDEOS and registro.esta_procesado(ruta_zip, nombre_video):
                            print("Este video ya ha sido procesado. Saltando...")
                            temp_dirs.append(None)
                            temp_paths.append(None)
                            continue
                        
                        # Extraer el video temporalmente
                        print("Extrayendo video temporalmente...")
                        temp_dir, ruta_temp = extraer_video_temporal(ruta_zip, nombre_video)
                        
                        if not temp_dir or not ruta_temp:
                            print("No se pudo extraer el video temporalmente")
                            temp_dirs.append(None)
                            temp_paths.append(None)
                            continue
                            
                        print(f"Video extraído en: {ruta_temp}")
                        temp_dirs.append(temp_dir)
                        temp_paths.append(ruta_temp)
                    
                    # Ahora procesar cada video extraído
                    for k, video_info in enumerate(batch_videos):
                        if temp_dirs[k] is None:
                            continue  # Saltar videos que no se pudieron extraer
                            
                        nombre_video = video_info["nombre"]
                        ruta_temp = temp_paths[k]
                        
                        print(f"\n[{j+k+1}/{len(videos_filtrados)}] Procesando: {nombre_video}")
                        
                        try:
                            # Procesar el video con YOLOv8
                            print("Procesando video con YOLOv8...")
                            
                            # Obtener nombre base del video (sin extensión ni ruta)
                            nombre_base = os.path.splitext(os.path.basename(nombre_video))[0]
                            
                            # Ajustar el comando YOLOv8 para usar el modelo específico del disco
                            # y evitar descargas automáticas que podrían fallar
                            yolo_model_path = os.path.join(os.getcwd(), "yolov8s.pt")
                            if os.path.exists(yolo_model_path):
                                modelo = yolo_model_path
                            else:
                                # Si no existe, usar un path relativo (asumiendo que está en el directorio actual)
                                modelo = "yolov8s.pt"
                                
                            # CAMBIO IMPORTANTE: Ahora especificamos explícitamente resultados_dir como 
                            # ubicación de salida para el Excel en el script de YOLO
                            # Crear directorio específico para este video
                            video_results_dir = os.path.join(resultados_dir, nombre_base)
                            os.makedirs(video_results_dir, exist_ok=True)
                            
                            # Modificar la variable de entorno PYTHONPATH para garantizar que ultralytics
                            # use nuestro directorio de resultados en lugar del predeterminado
                            env = os.environ.copy()
                            env["PYTHONPATH"] = f"{os.getcwd()}:{env.get('PYTHONPATH', '')}"
                            
                            # Especificar explícitamente la ruta de resultados
                            comando_shell = f'python yolov8-object-tracking/yolo/v8/detect/detect_follow.py model="{modelo}" source="{ruta_temp}" save=True project={video_results_dir}'
                            print(f"Ejecutando comando con shell=True: {comando_shell}")
                            
                            # NUEVO: Guardar la salida del comando para análisis
                            resultado = subprocess.run(comando_shell, shell=True, check=True, 
                                                     stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                                                     text=True, env=env)
                            
                            # Imprimir salida
                            print("SALIDA ESTÁNDAR:")
                            print(resultado.stdout)
                            if resultado.stderr:
                                print("ERRORES:")
                                print(resultado.stderr)
                                
                            # NUEVO: Analizar la salida para extraer el directorio de resultados
                            results_dir = verificar_resultados_yolo(nombre_video, resultado.stdout + resultado.stderr)

                            # Buscar el archivo Excel de resultados
                            excel_path = encontrar_excel_resultado(nombre_video, video_results_dir)
                            
                            # Si no se encuentra el Excel en nuestro directorio, buscar en otros lugares
                            if not excel_path:
                                print("No se encontró el Excel en el directorio especificado, buscando en otras ubicaciones...")
                                excel_path = encontrar_excel_resultado(nombre_video, None)
                            
                            if excel_path and os.path.exists(excel_path):
                                print(f"Archivo de resultados encontrado: {excel_path}")
                                
                                # NUEVO: Si el Excel está en otra ubicación, copiarlo a nuestro directorio de resultados
                                if not excel_path.startswith(resultados_dir):
                                    nuevo_excel_path = os.path.join(video_results_dir, f"{nombre_base}_resultados.xlsx")
                                    try:
                                        shutil.copy2(excel_path, nuevo_excel_path)
                                        print(f"Excel copiado al directorio de resultados: {nuevo_excel_path}")
                                        excel_path = nuevo_excel_path
                                    except Exception as e:
                                        print(f"Error al copiar Excel: {e}")
                                
                                # Marcar como procesado en el registro
                                registro.marcar_como_procesado(ruta_zip, nombre_video, "completado", excel_path)
                                
                                # Analizar resultados si está habilitado
                                if ANALYZE_RESULTS:
                                    print("\nAnalizando resultados de detección...")
                                    success = analyze_excel_file(excel_path, ruta_temp)
                                    
                                    if not success:
                                                                          
                                    print("No se encontró un archivo Excel con resultados reales. Continuando con el siguiente video...")
                                    # Marcar como procesado pero con estado de error
                                    registro.marcar_como_procesado(ruta_zip, nombre_video, "error: no se generó archivo de resultados")
                                else:
                                    print(f"No se encontró el archivo Excel de resultados para {nombre_video}")
                                    print("Se requieren resultados reales para el análisis. Continuando con el siguiente video...")
                                    registro.marcar_como_procesado(ruta_zip, nombre_video, "error: no se encontró archivo de resultados")
                                
                        except subprocess.CalledProcessError as e:
                            print(f"Error al ejecutar YOLOv8: {e}")
                            if hasattr(e, 'stderr'):
                                print(f"Salida de error: {e.stderr}")
                            if hasattr(e, 'stdout'):
                                print(f"Salida estándar: {e.stdout}")
                            registro.marcar_como_procesado(ruta_zip, nombre_video, "error")
                        except Exception as e:
                            print(f"Error al procesar el video: {str(e)}")
                            import traceback
                            traceback.print_exc()
                            registro.marcar_como_procesado(ruta_zip, nombre_video, "error")
                            
                finally:
                    # Limpiar archivos temporales después de procesar el lote completo
                    print("\nLimpiando archivos temporales del lote...")
                    for temp_dir in temp_dirs:
                        if temp_dir:
                            try:
                                shutil.rmtree(temp_dir)
                            except Exception as e:
                                print(f"Error al eliminar directorio temporal: {str(e)}")
                    
                    print(f"Lote {j//batch_size + 1} completado")
        
        # Mostrar estadísticas finales
        estadisticas_finales = registro.obtener_estadisticas()
        print("\n" + "="*50)
        print("ESTADÍSTICAS FINALES")
        print("="*50)
        print(f"Total de videos procesados: {estadisticas_finales['total_videos']}")
        print(f"Videos completados: {estadisticas_finales['completados']}")
        print(f"Videos con errores: {estadisticas_finales['con_errores']}")
        print(f"Videos saltados: {estadisticas_finales['saltados']}")
        print(f"Última actualización: {estadisticas_finales['ultima_actualizacion']}")
        print("="*50)
    
    except Exception as e:
        print(f"\nError inesperado: {str(e)}")
        import traceback
        traceback.print_exc()
        
        
# Nueva función para buscar archivos Excel recientes en todo el sistema
def find_recent_excel_files(minutes=5):
    """
    Busca archivos Excel modificados en los últimos N minutos.
    
    Args:
        minutes (int): Número de minutos a considerar para la búsqueda
        
    Returns:
        list: Lista de tuplas (ruta_archivo, tiempo_modificacion) ordenada por más recientes
    """
    excel_files = []
    tiempo_actual = time.time()
    tiempo_limite = tiempo_actual - (minutes * 60)  # Convertir minutos a segundos
    
    directorios_busqueda = [
        os.getcwd(),
        os.path.join(os.getcwd(), "runs"),
        os.path.join(os.getcwd(), "runs/detect"),
        os.path.join(os.getcwd(), "resultados"),
        os.path.expanduser("~")
    ]
    
    print(f"Buscando archivos Excel modificados en los últimos {minutes} minutos...")
    
    for directorio in directorios_busqueda:
        if not os.path.exists(directorio):
            continue
            
        print(f"Buscando en: {directorio}")
        
        for ruta, _, archivos in os.walk(directorio):
            for archivo in archivos:
                if archivo.endswith('.xlsx'):
                    ruta_completa = os.path.join(ruta, archivo)
                    try:
                        tiempo_modificacion = os.path.getmtime(ruta_completa)
                        if tiempo_modificacion >= tiempo_limite:
                            excel_files.append((ruta_completa, tiempo_modificacion))
                    except Exception as e:
                        print(f"Error al verificar tiempo de {ruta_completa}: {e}")
    
    # Ordenar por tiempo de modificación (más reciente primero)
    excel_files.sort(key=lambda x: x[1], reverse=True)
    return excel_files

# Nueva función para crear un Excel mínimo con datos de ejemplo para análisis
def create_minimal_excel(excel_path, video_name):
    """
    Crea un archivo Excel con datos simulados más realistas para el análisis de trayectorias.
    Los datos se generan basados en dimensiones de cámara de tráfico típicas.
    """
    import pandas as pd
    import random
    import math
    from collections import Counter
    
    # Crear directorio para el Excel si no existe
    os.makedirs(os.path.dirname(excel_path), exist_ok=True)
    
    # Generar datos de ejemplo para vehículos
    video_id = os.path.splitext(os.path.basename(video_name))[0]
    video_id = video_id.replace("_manual", "")  # Eliminar "_manual" si existe
    
    # Usar el nombre del video como semilla para asegurar resultados consistentes por video
    seed_value = 0
    for char in video_id:
        seed_value += ord(char)
    random.seed(seed_value)
    
    # Número de vehículos realista para un video de tráfico corto
    num_vehicles = random.randint(15, 30)
    
    # Dimensiones típicas de un cuadro de video de cámara de tráfico
    frame_width, frame_height = 640, 480
    
    # Generamos un desplazamiento único para este video
    # para que diferentes videos tengan diferentes patrones
    offset_x = (hash(video_id) % 40) - 20  # Rango -20 a +20
    offset_y = (hash(video_id[::-1]) % 40) - 20
    
    # MEJORADO: Definir áreas de interés (ROIs) para simular intersecciones, acercándose
    # más a lo que se vería en un video de tráfico real
    # Definimos 4 carriles entrando a una intersección desde diferentes direcciones
    
    # Centro de la intersección (con variación basada en el video ID)
    center_x = frame_width / 2 + offset_x
    center_y = frame_height / 2 + offset_y
    
    # Longitud de los segmentos de carretera
    road_length = min(frame_width, frame_height) * 0.35
    
    # Definición de los 4 carriles principales (puntos iniciales y finales)
    roads = [
        # Carril desde arriba hacia el centro
        [(center_x, center_y - road_length), (center_x, center_y)],
        # Carril desde abajo hacia el centro
        [(center_x, center_y + road_length), (center_x, center_y)],
        # Carril desde la izquierda hacia el centro
        [(center_x - road_length, center_y), (center_x, center_y)],
        # Carril desde la derecha hacia el centro
        [(center_x + road_length, center_y), (center_x, center_y)]
    ]
    
    # Agregar variación a los carriles para simular diferentes anchos
    roads_with_variation = []
    for road in roads:
        # Variaciones basadas en el ID del video
        variation = random.uniform(0.8, 1.2)
        road_var = [(p[0] * variation, p[1]) for p in road]
        roads_with_variation.append(road_var)
    
    # Función para generar una trayectoria realista a lo largo de un carril
    def generate_trajectory(road_idx, num_points=5):
        # Obtener carril base
        start, end = roads[road_idx]
        
        # Generar puntos intermedios con variación para simular movimiento natural
        points = [start]
        for i in range(1, num_points-1):
            # Interpolación con ruido
            t = i / (num_points-1)
            mid_x = start[0] + (end[0] - start[0]) * t
            mid_y = start[1] + (end[1] - start[1]) * t
            
            # Agregar variación simulando conducción imperfecta
            mid_x += random.gauss(0, 5)  # Desviación estándar de 5 píxeles
            mid_y += random.gauss(0, 5)
            
            points.append((mid_x, mid_y))
        
        points.append(end)
        return points
    
    # Tipos de vehículos con distribución más realista
    vehicle_types = ['car'] * 70 + ['motorcycle'] * 15 + ['bus'] * 5 + ['truck'] * 10
    
    # Generar datos
    data = []
    for i in range(1, num_vehicles + 1):
        # Seleccionar un carril de origen aleatorio
        road_idx = random.randint(0, len(roads)-1)
        
        # Generar una trayectoria base
        trajectory = generate_trajectory(road_idx, num_points=random.randint(4, 7))
        
        # Agregar variación final para individualizar cada trayectoria
        trajectory = [(x + random.gauss(0, 3), y + random.gauss(0, 3)) for x, y in trajectory]
        
        # Formatear como string
        trajectory_str = " -> ".join([f"({int(x)},{int(y)})" for x, y in trajectory])
        
        # Seleccionar tipo de vehículo según distribución realista
        vehicle_type = random.choice(vehicle_types)
        
        # Añadir a datos
        data.append({
            'ID': i,
            'Tipo de Vehículo': vehicle_type,
            'Trayectoria': trajectory_str,
            'Tiempo': datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        })
    
    # Crear DataFrame y guardar
    df = pd.DataFrame(data)
    
    # Crear Excel
    with pd.ExcelWriter(excel_path) as writer:
        df.to_excel(writer, sheet_name='Datos_Detallados', index=False)
    
    print(f"Excel creado con {num_vehicles} vehículos simulados en: {excel_path}")
    return excel_path

# Add this function to your file - it's referenced but missing in your implementation
def verificar_resultados_yolo(nombre_video, script_output):
    """
    Analiza la salida del script de YOLO para identificar el directorio de resultados
    y buscar pistas sobre la ubicación del Excel.
    
    Args:
        nombre_video (str): Nombre del video procesado
        script_output (str): Salida del script de YOLO
        
    Returns:
        str: Ruta al directorio de resultados, o None si no se encuentra
    """
    # Buscar patrón "Results saved to **runs/detect/X**" en la salida
    pattern = r'Results saved to \*\*(.*?)\*\*'
    match = re.search(pattern, script_output)
    
    if match:
        results_dir = match.group(1)
        print(f"Se identificó el directorio de resultados de YOLO: {results_dir}")
        
        # Verificar si existe
        if os.path.exists(results_dir):
            print(f"El directorio de resultados existe")
            # Buscar archivos Excel en ese directorio
            for archivo in os.listdir(results_dir):
                if archivo.endswith('.xlsx'):
                    ruta_completa = os.path.join(results_dir, archivo)
                    print(f"Encontrado Excel en directorio de resultados: {ruta_completa}")
                    return results_dir
            
            print("No se encontraron archivos Excel en el directorio de resultados")
        else:
            print(f"El directorio de resultados no existe: {results_dir}")
    else:
        print("No se pudo identificar el directorio de resultados en la salida del script")
    
    return None

def main():
    # Define aquí la función que hace la exportación
    def export_function():
        return export_to_excel(output_dir, video_name)
    
    # Define la condición para verificar si hay datos
    def hay_datos():
        return len(vehicle_data) > 0
    
    # Configura los manejadores
    setup_export_handlers(export_function, hay_datos)
    
    # Resto de tu código main()...

def main():
    print("\n" + "="*50)
    print("SISTEMA DE ANÁLISIS DE VIDEOS AUTOMATIZADO")
    print("="*50)
        
    try:
        # Procesar archivos ZIP automáticamente con límite de duración de 2 minutos (120 segundos)
        procesar_archivos_zip_automatizado(batch_size=5, max_duration_seconds=300)
        
        print("\nProcesamiento automático completado.")
    except KeyboardInterrupt:
        # El handler ya maneja la exportación
        print("\nPrograma interrumpido por el usuario.")
    except Exception as e:
        print(f"\nError inesperado: {str(e)}")
        # Exportar en caso de error
        if len(vehicle_data) > 0:
            export_to_excel(output_dir, video_name)
    finally:
        print("\nPrograma finalizado.")


if __name__ == "__main__":
    main()

IndentationError: expected an indented block after 'if' statement on line 1390 (920761579.py, line 1392)

In [ ]:
!pip install torch==2.5.1 torchvision torchaudio

ERROR: Could not find a version that satisfies the requirement torch==2.5.1 (from versions: 2.6.0, 2.7.0)
ERROR: No matching distribution found for torch==2.5.1


In [26]:
# In your Jupyter notebook, run:
with open('run_yolo.py', 'w') as f:
    f.write('''
"""
YOLO Wrapper Script for PyTorch 2.6+ Compatibility

This script modifies torch.load behavior to handle the serialization restrictions
introduced in PyTorch 2.6, allowing YOLO model weights to load correctly.
"""

import sys
import os
import torch

# Add the necessary classes to the safe globals list for PyTorch 2.6+
# This is based on the specific error message we received
torch.serialization.add_safe_globals([
    'ultralytics.nn.tasks.DetectionModel',
    'ultralytics.nn.modules.Conv',
    'ultralytics.nn.modules.Bottleneck',
    'ultralytics.nn.modules.C3',
    'ultralytics.nn.modules.SPPF',
    'ultralytics.nn.modules.Detect'
])

# Also monkey patch torch.load to handle the case where weights_only=True
# causes issues with the YOLO model
original_torch_load = torch.load

def patched_torch_load(f, map_location=None, pickle_module=None, **kwargs):
    """
    Patched version of torch.load that attempts both weights_only=False and True
    """
    try:
        # First try with weights_only=False (pre-2.6 behavior)
        return original_torch_load(f, map_location=map_location, 
                               pickle_module=pickle_module, 
                               weights_only=False, **kwargs)
    except Exception as e:
        print(f"Warning: Error loading with weights_only=False: {e}")
        print("Retrying with weights_only=True and safe globals...")
        # If that fails, try with weights_only=True and our safe globals
        return original_torch_load(f, map_location=map_location, 
                               pickle_module=pickle_module, 
                               weights_only=True, **kwargs)

# Replace torch.load with our patched version
torch.load = patched_torch_load

# Get the command line arguments for the YOLO script
yolo_script = 'yolov8-object-tracking/yolo/v8/detect/detect_follow.py'
args = sys.argv[1:]  # Skip the script name (run_yolo.py)

# Build the command to run the YOLO script
cmd_args = ' '.join(f'"{arg}"' if ' ' in arg else arg for arg in args)
full_cmd = f'python {yolo_script} {cmd_args}'

print(f"Running command with PyTorch 2.6+ compatibility: {full_cmd}")

# Run the YOLO script
exit_code = os.system(full_cmd)
sys.exit(exit_code // 256)  # Convert os.system exit code to regular exit code
''')
print("Created improved wrapper script: run_yolo.py")

Created improved wrapper script: run_yolo.py


In [27]:
import subprocess

try:
    result = subprocess.run(['python', 'run_yolo.py', '--help'], 
                           capture_output=True, text=True, check=True)
    print("Wrapper script test successful!")
    print("\nOutput:")
    print(result.stdout)
except subprocess.CalledProcessError as e:
    print(f"Error running wrapper script: {e}")
    print("\nStderr:")
    print(e.stderr)

Wrapper script test successful!

Output:
Running command with PyTorch 2.6+ compatibility: python yolov8-object-tracking/yolo/v8/detect/detect_follow.py --help
detect_follow is powered by Hydra.

== Configuration groups ==
Compose your configuration from those groups (group=option)



== Config ==
Override anything in the config (foo.bar=value)

task: detect
mode: train
model: null
data: null
epochs: 100
patience: 50
batch: 16
imgsz: 640
save: true
cache: false
device: null
workers: 8
project: null
name: null
exist_ok: false
pretrained: false
optimizer: SGD
verbose: false
seed: 0
deterministic: true
single_cls: false
image_weights: false
rect: false
cos_lr: false
close_mosaic: 10
resume: false
overlap_mask: true
mask_ratio: 4
dropout: 0.0
val: true
save_json: false
save_hybrid: false
conf: null
iou: 0.7
max_det: 300
half: false
dnn: false
plots: true
source: null
show: false
save_txt: false
save_conf: false
save_crop: false
hide_labels: false
hide_conf: false
vid_stride: 1
line_thicknes

In [28]:
import os
import cv2
import zipfile
import tempfile
import shutil
import json
import datetime
import subprocess
import re
import math
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from collections import Counter
import pandas as pd
import time
from pathlib import Path

BASE_DIRECTORY = "/Volumes/Elements/Daniela"  # hay que cambiar esto al path correcto
SHOW_DURATION = False  # false para que no imprima la duracion de los videos
PROCESS_ALL_VIDEOS = True  # vamos a procesar todos los videos
ANALYZE_RESULTS = True  # analizar los resultados de la detección de vehículos inmediatamente después
RESULTS_DIRECTORY = "runs/detect"  # Directorio donde YOLO guarda los resultados

# NUEVA CONFIGURACIÓN PARA LA ESTRUCTURA DE DIRECTORIOS
NUEVA_ESTRUCTURA = True  # Activar la nueva estructura resultados > ID > fecha
EXCEL_IDS_PATH = "ID_Camaras.xlsx"  # Ruta al Excel con los IDs de cámaras (si es None, se usan IDs hardcodeados)

# Lista de IDs predefinidos (se usa si EXCEL_IDS_PATH es None)
CAMERA_IDS = [
    "0*1_00DE1A7BF46241A4AC150B4EDD5CE7E8_", "0*1_0BCA0B2C1F8B4608831C622328453A09_",
    "0*1_25EF106A732747589666225E833039FD_", "0*1_28E98E2D3CA543B5B61A5372DE1D6176_",
    "0*1_349EC3DEF619474DB4954A3AC456CC44_", "0*1_4E29D1CE35C14CD8885B5446C176471A_",
    "0*1_525D839C1A81403297332790A344B8A3_", "0*1_554FE21AB49D4CB0904419E64D5CFCA7_",
    "0*1_5B693E05A5444F6FAA12EB4DCAD38FE1_", "0*1_688B53BEEA7246D4BE0E7793CFE69A3A_",
    "0*1_6B66BEBEE25F435392597505F536A3C4_", "0*1_7947BE58335A4D6598B53AAE1E352130_",
    "0*1_7977FAA000DF44299BDA3CB40F1DE5E2_", "0*1_79D91D550CED45C08BB2C2E744F6912E_",
    "0*1_79E4A01601CB4508993D8EC46B7713EA_", "0*1_7AF7702995734F638CCE2D5F8E28A399_",
    "0*1_7BC9BB67FD92413D9C05AAE59A91C351_", "0*1_8413AB94C90D4A148BC42CF8B8B0F6BC_",
    "0*1_8F80388561534015BE0131A385D922AD_", "0*1_908CC4463F2644169F76D75200F78426_",
    "0*1_92246844867A41EBA40D5AE096D692BC_", "0*1_93329799161242DDB3F9382B1DA696D7_",
    "0*1_AA2BFFD66A494EBFB296487F589F725E_", "0*1_AB87B86AB42D41E984F1ADAAA60EC11E_",
    "0*1_B3EA4269747A47C9B7A431D4EE18FC43_", "0*1_B446C3D57F324FAEBDD9878C18F4B6D6_",
    "0*1_BAD079BE592F4EF1BD7FE4ECC885C9E7_", "0*1_C345EE72F0B64951B95B8E998F3D398E_",
    "0*1_CD654999DD954EF9B22DC7CC75FF73D7_", "0*1_D161C43B16CB49359CC0E33EBF03F4E5_",
    "0*1_DDBF9DCF92F741EFB04D638F9DB388C9_", "0*1_E1BFBF6BDF144624B93FBDD30E0EF2E0_",
    "0*1_EBD46000BD6D42C997643C4B2E29A610_", "0*1_ECD961BE3E8F495E9F2DB778695E4050_",
    "0*1_EE0E7C3619654F77960059B3A676AA5D_", "0*1_F0F929BEFA854DC8997B44C1FF36B5C1_",
    "0*1_F2DDB53CE7334B4A84C939ABE3170A09_", "0*1_F7D6E4FF194349CE9B234CB7C4E258C0_",
    "0*1_F8A0D7A451CB477FB6127A2755166709_", "0*1_FB88E7DEE41B4F3CA49D6B6D63E1345D_",
    "0*1_FC551E9C8FD94BA6B2E2F62F491F9052_"
]

# Nueva función para cargar IDs de cámara desde un Excel
def cargar_ids_desde_excel(excel_path, columna_id='ID'):
    """
    Carga los IDs de cámara desde un archivo Excel.
    
    Args:
        excel_path (str): Ruta al archivo Excel
        columna_id (str): Nombre de la columna que contiene los IDs
        
    Returns:
        list: Lista de IDs de cámara
    """
    try:
        df = pd.read_excel(excel_path)
        if columna_id not in df.columns:
            print(f"Error: No se encontró la columna '{columna_id}' en el Excel")
            print(f"Columnas disponibles: {df.columns.tolist()}")
            return []
            
        # Extraer IDs y convertir a formato de string
        ids = df[columna_id].astype(str).tolist()
        
        # Limpiar IDs (eliminar espacios, NaN, etc.)
        ids = [id_cam.strip() for id_cam in ids if isinstance(id_cam, str) and id_cam.strip()]
        
        print(f"Se cargaron {len(ids)} IDs de cámara desde {excel_path}")
        return ids
    except Exception as e:
        print(f"Error al cargar IDs desde Excel: {str(e)}")
        return []

# Función para formatear tamaño de archivos
def formatear_tamaño(bytes, sufijo="B"):
    factor = 1024
    for unidad in ["", "K", "M", "G", "T", "P"]:
        if bytes < factor:
            return f"{bytes:.2f} {unidad}{sufijo}"
        bytes /= factor
    return f"{bytes:.2f} E{sufijo}"

# Función para obtener la duración de un video
def obtener_duracion_video(ruta_video):
    """
    Obtiene la duración de un video en segundos y en formato hora:minuto:segundo
    
    Args:
        ruta_video (str): Ruta al archivo de video
        
    Returns:
        tuple: (duracion_segundos, duracion_formateada)
    """
    try:
        # Abrir el video
        cap = cv2.VideoCapture(ruta_video)
        
        # Verificar si se abrió correctamente
        if not cap.isOpened():
            return None, "No disponible"
        
        # Obtener el número total de frames
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        # Obtener los frames por segundo (FPS)
        fps = cap.get(cv2.CAP_PROP_FPS)
        
        # Calcular la duración en segundos
        duracion_segundos = total_frames / fps if fps > 0 else 0
        
        # Formatear la duración en hora:minuto:segundo
        horas = int(duracion_segundos // 3600)
        minutos = int((duracion_segundos % 3600) // 60)
        segundos = int(duracion_segundos % 60)
        
        duracion_formateada = f"{horas:02d}:{minutos:02d}:{segundos:02d}"
        
        # Liberar el objeto de captura
        cap.release()
        
        return duracion_segundos, duracion_formateada
    except Exception as e:
        print(f"Error al obtener la duración del video: {str(e)}")
        return None, "Error"

# Función para identificar el ID de cámara en un nombre de archivo
def identificar_id_camara(nombre_archivo, lista_ids):
    """
    Identifica el ID de cámara en un nombre de archivo.
    
    Args:
        nombre_archivo (str): Nombre del archivo o ruta completa
        lista_ids (list): Lista de IDs de cámara a buscar
        
    Returns:
        str: ID de cámara encontrado o None si no se encuentra
    """
    if not lista_ids:
        print("Advertencia: Lista de IDs de cámara vacía")
        return None
    
    # Asegurarse de que estamos trabajando con el nombre completo
    nombre_completo = nombre_archivo
    print(f"Analizando nombre: {nombre_completo}")
    
    # Patrones de ID que podríamos encontrar
    patrones = [
        r'_0_1_[A-F0-9]+_',  # Formato _0_1_HEXID_
        r'\*0\*1_[A-F0-9]+_'  # Formato *0*1_HEXID_
    ]
    
    # Buscar IDs por patrones en el nombre del archivo
    for patron in patrones:
        coincidencia = re.search(patron, nombre_completo)
        if coincidencia:
            id_encontrado = coincidencia.group(0)
            print(f"ID encontrado por patrón '{patron}': {id_encontrado}")
            
            # Intentar normalizar el ID encontrado para compararlo con la lista
            id_normalizado = id_encontrado.replace('_0_1_', '*0*1_').replace('_0_1_', '*0*1_')
            
            # Verificar si existe en la lista (normalizado)
            for id_lista in lista_ids:
                id_lista_norm = id_lista.replace('*0*1_', '*0*1_')
                if id_normalizado.replace('_', '') == id_lista_norm.replace('_', ''):
                    print(f"ID coincide con entrada en lista: {id_lista}")
                    return id_lista
            
            # Si no está en la lista pero tiene formato válido, usarlo directamente
            return id_encontrado
    
    # Buscar por separadores (slash)
    partes_slash = nombre_completo.split('/')
    if len(partes_slash) > 1:
        primera_parte = partes_slash[0]
        for patron in patrones:
            coincidencia = re.search(patron, primera_parte)
            if coincidencia:
                id_encontrado = coincidencia.group(0)
                print(f"ID encontrado en parte antes de slash: {id_encontrado}")
                return id_encontrado
    
    # Buscar directamente en la lista de IDs
    for id_camara in lista_ids:
        # Intentar variaciones del formato
        variaciones = [
            id_camara,
            id_camara.replace('*0*1_', '_0_1_'),
            id_camara.replace('*', '_'),
            id_camara.replace('_', '')
        ]
        
        for variacion in variaciones:
            if variacion in nombre_completo:
                print(f"ID de cámara encontrado (coincidencia en lista): {id_camara}")
                return id_camara
    
    # Último recurso: Extraer cualquier secuencia que se parezca a un ID
    for patron in patrones:
        todas_coincidencias = re.findall(patron, nombre_completo)
        if todas_coincidencias:
            print(f"ID extraído como último recurso: {todas_coincidencias[0]}")
            return todas_coincidencias[0]
    
    print("No se pudo identificar el ID de cámara")
    return None

# Función para extraer fecha del nombre de archivo
def extraer_fecha_de_nombre(nombre_archivo):
    """
    Extrae la fecha de un nombre de archivo, buscando patrones comunes.
    
    Args:
        nombre_archivo (str): Nombre del archivo
        
    Returns:
        str: Fecha en formato YYYY-MM-DD o None si no se encuentra
    """
    # Buscar patrón YYYYMMDD
    patron_fecha = r'20\d{2}[01]\d[0-3]\d'
    coincidencia = re.search(patron_fecha, nombre_archivo)
    
    if coincidencia:
        fecha_str = coincidencia.group(0)
        try:
            fecha = datetime.datetime.strptime(fecha_str, "%Y%m%d")
            return fecha.strftime("%Y-%m-%d")
        except ValueError:
            pass
    
    # Otros patrones de fecha podrían agregarse aquí
    
    return None

# Función para formatear tamaño de archivos
def formatear_tamaño(bytes, sufijo="B"):
    factor = 1024
    for unidad in ["", "K", "M", "G", "T", "P"]:
        if bytes < factor:
            return f"{bytes:.2f} {unidad}{sufijo}"
        bytes /= factor
    return f"{bytes:.2f} E{sufijo}"

# Función para obtener la duración de un video
def obtener_duracion_video(ruta_video):
    """
    Obtiene la duración de un video en segundos y en formato hora:minuto:segundo
    
    Args:
        ruta_video (str): Ruta al archivo de video
        
    Returns:
        tuple: (duracion_segundos, duracion_formateada)
    """
    try:
        # Abrir el video
        cap = cv2.VideoCapture(ruta_video)
        
        # Verificar si se abrió correctamente
        if not cap.isOpened():
            return None, "No disponible"
        
        # Obtener el número total de frames
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        # Obtener los frames por segundo (FPS)
        fps = cap.get(cv2.CAP_PROP_FPS)
        
        # Calcular la duración en segundos
        duracion_segundos = total_frames / fps if fps > 0 else 0
        
        # Formatear la duración en hora:minuto:segundo
        horas = int(duracion_segundos // 3600)
        minutos = int((duracion_segundos % 3600) // 60)
        segundos = int(duracion_segundos % 60)
        
        duracion_formateada = f"{horas:02d}:{minutos:02d}:{segundos:02d}"
        
        # Liberar el objeto de captura
        cap.release()
        
        return duracion_segundos, duracion_formateada
    except Exception as e:
        print(f"Error al obtener la duración del video: {str(e)}")
        return None, "Error"

# Clase para gestionar el registro de videos procesados
class RegistroVideos:
    def __init__(self, ruta_registro=None):
        # Si no se especifica una ruta, usar el directorio del usuario
        if ruta_registro is None:
            self.ruta_registro = os.path.expanduser("~/videos_procesados.json")
        else:
            self.ruta_registro = ruta_registro
        
        # Inicializar o cargar el registro
        self.registro = self._cargar_registro()
    
    def _cargar_registro(self):
        # Crear el archivo si no existe
        if not os.path.exists(self.ruta_registro):
            # Crear un registro vacío
            registro_inicial = {
                "videos_procesados": {},
                "ultima_actualizacion": datetime.datetime.now().isoformat()
            }
            
            # Guardar el registro inicial
            with open(self.ruta_registro, 'w') as f:
                json.dump(registro_inicial, f, indent=2)
                
            return registro_inicial
        
        # Cargar el registro existente
        try:
            with open(self.ruta_registro, 'r') as f:
                return json.load(f)
        except Exception as e:
            print(f"Error al cargar el registro: {str(e)}")
            return {"videos_procesados": {}, "ultima_actualizacion": datetime.datetime.now().isoformat()}
    
    def esta_procesado(self, ruta_zip, nombre_video):
        # Crear un identificador único para el video
        id_video = f"{ruta_zip}:{nombre_video}"
        
        # Verificar si el video está en el registro
        return id_video in self.registro["videos_procesados"]
    
    def marcar_como_procesado(self, ruta_zip, nombre_video, resultado="completado", excel_path=None):
        # Crear un identificador único para el video
        id_video = f"{ruta_zip}:{nombre_video}"
        
        # Agregar el video al registro
        self.registro["videos_procesados"][id_video] = {
            "ruta_zip": ruta_zip,
            "nombre_video": nombre_video,
            "fecha_procesamiento": datetime.datetime.now().isoformat(),
            "resultado": resultado,
            "excel_path": excel_path
        }
        
        # Actualizar la fecha de última actualización
        self.registro["ultima_actualizacion"] = datetime.datetime.now().isoformat()
        
        # Guardar el registro actualizado
        self._guardar_registro()
    
    def _guardar_registro(self):
        try:
            with open(self.ruta_registro, 'w') as f:
                json.dump(self.registro, f, indent=2)
        except Exception as e:
            print(f"Error al guardar el registro: {str(e)}")
    
    def obtener_estadisticas(self):
        # Obtener estadísticas del registro
        total_videos = len(self.registro["videos_procesados"])
        completados = sum(1 for v in self.registro["videos_procesados"].values() if v["resultado"] == "completado")
        con_errores = sum(1 for v in self.registro["videos_procesados"].values() if v["resultado"] == "error")
        saltados = sum(1 for v in self.registro["videos_procesados"].values() if v["resultado"] == "saltado")
        
        return {
            "total_videos": total_videos,
            "completados": completados,
            "con_errores": con_errores,
            "saltados": saltados,
            "ultima_actualizacion": self.registro["ultima_actualizacion"]
        }
    
    def obtener_videos_procesados_por_zip(self, ruta_zip):
        # Obtener los videos procesados para un ZIP específico
        videos_procesados = []
        
        for id_video, info in self.registro["videos_procesados"].items():
            if info["ruta_zip"] == ruta_zip:
                videos_procesados.append(info["nombre_video"])
        
        return videos_procesados
    
    def obtener_excel_path(self, ruta_zip, nombre_video):
        # Crear un identificador único para el video
        id_video = f"{ruta_zip}:{nombre_video}"
        
        # Verificar si el video está en el registro y tiene una ruta de Excel asociada
        if id_video in self.registro["videos_procesados"]:
            return self.registro["videos_procesados"][id_video].get("excel_path")
        
        return None

# Función para listar videos dentro de un archivo ZIP
def listar_videos_en_zip(ruta_zip, obtener_duracion=False):
    """
    Lista los videos dentro de un archivo ZIP
    
    Args:
        ruta_zip (str): Ruta al archivo ZIP
        obtener_duracion (bool): Si es True, extrae y obtiene la duración de cada video
        
    Returns:
        list: Lista de diccionarios con información de cada video
    """
    videos = []
    extensiones_video = ['.mp4', '.avi', '.mov', '.mkv', '.wmv', '.flv']
    
    try:
        with zipfile.ZipFile(ruta_zip, 'r') as zip_ref:
            for archivo in zip_ref.namelist():
                if any(archivo.lower().endswith(ext) for ext in extensiones_video):
                    info_video = {
                        "nombre": archivo,
                        "tamaño": formatear_tamaño(zip_ref.getinfo(archivo).file_size),
                        "duracion_segundos": None,
                        "duracion": "No extraído"
                    }
                    
                    # Si se requiere la duración, extraer el video temporalmente
                    if obtener_duracion:
                        temp_dir, ruta_temp = extraer_video_temporal(ruta_zip, archivo)
                        
                        if temp_dir and ruta_temp:
                            try:
                                duracion_segundos, duracion = obtener_duracion_video(ruta_temp)
                                info_video["duracion_segundos"] = duracion_segundos
                                info_video["duracion"] = duracion
                            finally:
                                # Limpiar archivos temporales
                                try:
                                    shutil.rmtree(temp_dir)
                                except Exception as e:
                                    print(f"Error al eliminar archivos temporales: {str(e)}")
                    
                    videos.append(info_video)
    except Exception as e:
        print(f"Error al leer {ruta_zip}: {str(e)}")
    
    return videos

# Función para encontrar archivos ZIP
def encontrar_archivos_zip(directorio_base):
    archivos_zip = []
    
    for carpeta_actual, _, archivos in os.walk(directorio_base):
        for archivo in archivos:
            if archivo.lower().endswith('.zip'):
                ruta_completa = os.path.join(carpeta_actual, archivo)
                archivos_zip.append(ruta_completa)
    
    return archivos_zip

# Función para extraer un solo video del ZIP a un archivo temporal
def extraer_video_temporal(ruta_zip, nombre_video):
    try:
        # Crear un directorio temporal
        temp_dir = tempfile.mkdtemp()
        
        # Extraer solo el archivo de video específico
        with zipfile.ZipFile(ruta_zip, 'r') as zip_ref:
            zip_ref.extract(nombre_video, temp_dir)
        
        # Ruta completa al archivo extraído
        ruta_temp = os.path.join(temp_dir, nombre_video)
        
        return temp_dir, ruta_temp
    except Exception as e:
        print(f"Error al extraer {nombre_video}: {str(e)}")
        return None, None

def encontrar_excel_resultado(nombre_video, results_dir=None, camera_id=None, fecha=None):
    """
    Busca el archivo Excel de resultados más reciente para un video procesado.
    Considera la nueva estructura de directorios: resultados > ID > fecha
    
    Args:
        nombre_video (str): Nombre del video
        results_dir (str): Directorio base de resultados
        camera_id (str): ID de la cámara (para nueva estructura)
        fecha (str): Fecha en formato YYYY-MM-DD (para nueva estructura)
    """
    import os
    import datetime
    import time
    
    # Obtener nombre base del video (sin extensión y sin ruta)
    nombre_base = os.path.splitext(os.path.basename(nombre_video))[0]
    print(f"Buscando archivos Excel para video con nombre base: {nombre_base}")
    
    # Lista de directorios a buscar
    directorios_busqueda = [
        RESULTS_DIRECTORY,              # El directorio configurado en las variables globales
        "runs/detect",                  # Directorio predeterminado de YOLO
        "runs/detect/train3",           # El directorio que aparece en los mensajes de salida
        os.path.join(os.getcwd(), "runs/detect"),  # Ruta absoluta
        os.path.join(os.getcwd(), "runs/detect/train3"),
        os.path.join(os.getcwd(), "resultados"),   # Directorio donde el script YOLO guarda resultados
        os.path.expanduser("~/resultados"),        # Por si está guardando en el directorio del usuario
    ]
    
    # Agregar directorio específico si se proporciona
    if results_dir:
        directorios_busqueda.insert(0, results_dir)  # Prioridad más alta
    
    # Agregar directorios para la nueva estructura
    if NUEVA_ESTRUCTURA and camera_id:
        # Construir rutas con la nueva estructura
        for dir_base in directorios_busqueda.copy():
            if fecha:
                # resultados > ID > fecha
                nueva_ruta = os.path.join(dir_base, camera_id, fecha)
                directorios_busqueda.insert(0, nueva_ruta)  # Mayor prioridad
            
            # resultados > ID (por si no hay subdirectorio de fecha)
            nueva_ruta_id = os.path.join(dir_base, camera_id)
            directorios_busqueda.insert(0, nueva_ruta_id)
    
    print(f"Buscando en los siguientes directorios: {directorios_busqueda}")
    
    # Lista para almacenar todos los Excel encontrados
    excel_encontrados = []
    
    # Buscar en cada directorio
    for directorio in directorios_busqueda:
        if not os.path.exists(directorio):
            print(f"Directorio no encontrado: {directorio}")
            continue
            
        print(f"Buscando en directorio: {directorio}")
        
        # Buscar archivos Excel en este directorio
        for carpeta_actual, _, archivos in os.walk(directorio):
            for archivo in archivos:
                if archivo.endswith('.xlsx'):
                    ruta_completa = os.path.join(carpeta_actual, archivo)
                    
                    # Verificar si el nombre base está en el nombre del archivo
                    if nombre_base in archivo:
                        print(f"¡Encontrado archivo Excel con coincidencia exacta!: {ruta_completa}")
                        tiempo_modificacion = os.path.getmtime(ruta_completa)
                        excel_encontrados.append((ruta_completa, tiempo_modificacion, 1))  # Prioridad 1 (alta)
                    else:
                        # Archivos Excel sin coincidencia exacta tienen menor prioridad
                        tiempo_modificacion = os.path.getmtime(ruta_completa)
                        excel_encontrados.append((ruta_completa, tiempo_modificacion, 2))  # Prioridad 2 (baja)
    
    # Busca Excel modificados en los últimos 5 minutos (para capturar archivos recién creados)
    tiempo_actual = time.time()
    for directorio in directorios_busqueda:
        if not os.path.exists(directorio):
            continue
            
        for carpeta_actual, _, archivos in os.walk(directorio):
            for archivo in archivos:
                if archivo.endswith('.xlsx'):
                    ruta_completa = os.path.join(carpeta_actual, archivo)
                    tiempo_modificacion = os.path.getmtime(ruta_completa)
                    
                    # Si fue modificado en los últimos 5 minutos, considerarlo de alta prioridad
                    if tiempo_actual - tiempo_modificacion < 300:  # 300 segundos = 5 minutos
                        # Verificar si ya está en la lista
                        if not any(ruta_completa == item[0] for item in excel_encontrados):
                            print(f"¡Encontrado archivo Excel reciente!: {ruta_completa}")
                            excel_encontrados.append((ruta_completa, tiempo_modificacion, 0))  # Prioridad 0 (más alta)
    
    # Ordenar por prioridad (primero) y tiempo de modificación (segundo criterio)
    excel_encontrados.sort(key=lambda x: (x[2], -x[1]))
    
    # Imprimir todos los archivos encontrados
    if excel_encontrados:
        print(f"Archivos Excel encontrados (ordenados por prioridad y más recientes):")
        for idx, (path, time, priority) in enumerate(excel_encontrados[:5]):  # Mostrar los 5 más relevantes
            print(f"  {idx+1}. {path} (modificado: {datetime.datetime.fromtimestamp(time)}, prioridad: {priority})")
        
        # Devolver el más prioritario
        return excel_encontrados[0][0]
    
    print("No se encontró ningún archivo Excel relacionado con el video")
    return None

def guardar_archivo_nueva_estructura(ruta_origen, directorio_resultados, camera_id, fecha=None):
    """
    Guarda un archivo en la nueva estructura de directorios.
    
    Args:
        ruta_origen (str): Ruta al archivo original
        directorio_resultados (str): Directorio base de resultados
        camera_id (str): ID de la cámara
        fecha (str): Fecha en formato YYYY-MM-DD (opcional)
        
    Returns:
        str: Ruta al archivo guardado o None si hay error
    """
    try:
        if not os.path.exists(ruta_origen):
            print(f"Error: El archivo origen no existe: {ruta_origen}")
            return None
        
        # Verificar que el ID no esté vacío
        if not camera_id:
            print("Error: Se requiere un ID de cámara válido")
            camera_id = "unknown_camera"  # Usar un valor predeterminado
            
        # Obtener nombre del archivo
        nombre_archivo = os.path.basename(ruta_origen)
        
        # Si no se proporciona fecha, intentar extraerla del nombre o usar la actual
        if not fecha:
            fecha = extraer_fecha_de_nombre(nombre_archivo)
            if not fecha:
                fecha = datetime.datetime.now().strftime("%Y-%m-%d")
        
        # Crear estructura de directorios
        directorio_camara = os.path.join(directorio_resultados, camera_id)
        print(f"Creando directorio de cámara: {directorio_camara}")
        os.makedirs(directorio_camara, exist_ok=True)
        
        # Crear directorio por fecha
        directorio_fecha = os.path.join(directorio_camara, fecha)
        print(f"Creando directorio de fecha: {directorio_fecha}")
        os.makedirs(directorio_fecha, exist_ok=True)
        
        # Ruta destino
        ruta_destino = os.path.join(directorio_fecha, nombre_archivo)
        
        # Copiar el archivo
        shutil.copy2(ruta_origen, ruta_destino)
        print(f"Archivo guardado en nueva estructura: {ruta_destino}")
        
        # Verificar que la copia fue exitosa
        if os.path.exists(ruta_destino):
            print(f"Verificación: El archivo se copió correctamente a {ruta_destino}")
            return ruta_destino
        else:
            print(f"Error: La verificación de copia falló para {ruta_destino}")
            return None
    except Exception as e:
        print(f"Error al guardar en nueva estructura: {str(e)}")
        import traceback
        traceback.print_exc()
        return None


# ===== FUNCIONES PARA ANÁLISIS DE TRAYECTORIAS =====

def determine_optimal_clusters(angle_data, min_clusters=2, max_clusters=6):
    """
    Determine optimal number of clusters using silhouette score.
    This helps find the natural groupings in the data.
    """
    if len(angle_data) < max_clusters:
        return min(len(angle_data), min_clusters)
    
    # Convert angles to points on a unit circle for proper clustering
    x = np.cos(np.radians(angle_data))
    y = np.sin(np.radians(angle_data))
    
    # Combine into feature array
    features = np.column_stack((x, y))
    
    best_score = -1
    best_n_clusters = min_clusters
    
    # Try different cluster counts and find the best one
    for n_clusters in range(min_clusters, min(max_clusters + 1, len(angle_data))):
        kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        cluster_labels = kmeans.fit_predict(features)
        
        # Calculate silhouette score
        try:
            score = silhouette_score(features, cluster_labels)
            print(f"  Testing {n_clusters} clusters: silhouette score = {score:.4f}")
            
            if score > best_score:
                best_score = score
                best_n_clusters = n_clusters
        except:
            # If silhouette score fails (e.g., only one sample in cluster)
            continue
    
    print(f"  Optimal number of clusters: {best_n_clusters} (score: {best_score:.4f})")
    return best_n_clusters


def process_vehicle_data(file_path, sheet_name="Datos_Detallados", min_clusters=2, max_clusters=6, min_displacement=5):
    """Process vehicle data and classify directions with adaptive clustering."""
    print(f"Procesando análisis de trayectorias para: {file_path}...")
    
    # Try to extract video ID from filename
    video_id = os.path.splitext(os.path.basename(file_path))[0]
    print(f"Archivo: {video_id}")
    
    # Load data
    try:
        df = pd.read_excel(file_path, sheet_name=sheet_name)
    except Exception as e:
        print(f"Error al cargar archivo Excel: {e}")
        return None, None, None
    
    print(f"Cargados {len(df)} registros de detecciones de vehículos")
    
    # Skip header row if it's duplicated
    if "ID" in df.columns and df.iloc[0]["ID"] == "ID":
        df = df.iloc[1:].reset_index(drop=True)
        
    # Convert ID to numeric if it's not already
    if "ID" in df.columns:
        try:
            df["ID"] = pd.to_numeric(df["ID"])
            # Keep only the last record for each vehicle (final trajectory)
            df.drop_duplicates(subset="ID", keep="last", inplace=True)
            print(f"Después de eliminar duplicados: {len(df)} vehículos únicos")
        except Exception as e:
            print(f"Error al convertir ID a numérico: {e}")
        
    # Extract trajectory features
    try:
        # Find trajectory column
        traj_col = None
        for col in df.columns:
            if 'trayectoria' in col.lower() or 'trajectory' in col.lower():
                traj_col = col
                break
        
        if not traj_col:
            print("Error: No se pudo encontrar la columna de trayectoria")
            return None, None, None
        
        # Parse trajectories
        df['trajectory_points'] = df[traj_col].apply(parse_trajectory)
        
        # Calculate direction features
        vectors = [calculate_trajectory_vector(points) for points in df['trajectory_points']]
        df['dx'] = [v[0] if v is not None and v[0] is not None else 0 for v in vectors]
        df['dy'] = [v[1] if v is not None and v[1] is not None else 0 for v in vectors]
        df['angle'] = [v[2] if v is not None and v[2] is not None else -1 for v in vectors]
        
        # Calculate displacement distance
        df['distance'] = np.sqrt(df['dx']**2 + df['dy']**2)
        
        # Filter out invalid trajectories (too short or no movement)
        valid_df = df[df['distance'] > min_displacement].copy()
        print(f"Encontradas {len(valid_df)} trayectorias válidas con movimiento suficiente")
        
        if len(valid_df) < min_clusters:
            print(f"Advertencia: No hay suficientes trayectorias válidas para agrupamiento. Se necesitan al menos {min_clusters}.")
            return df, None, None
            
        # Get valid angles for clustering
        valid_angles = valid_df[valid_df['angle'] >= 0]['angle'].values
        
        # Determine optimal number of clusters
        print("Determinando número óptimo de grupos (clusters)...")
        n_clusters = determine_optimal_clusters(valid_angles, min_clusters, max_clusters)
        
        # Perform clustering with optimal number of clusters
        print(f"Agrupando con {n_clusters} direcciones principales...")
        cluster_angles, valid_df = cluster_directions(valid_df, n_clusters)
        
        if cluster_angles is None:
            print("El agrupamiento falló.")
            return df, None, None
        
        # Name clusters
        cluster_to_name, sorted_angles = name_direction_clusters(cluster_angles, video_id)
        
        # Assign direction names
        valid_df = assign_direction_names(valid_df, cluster_to_name)
        
        # Print cluster information
        print("\nGrupos de dirección identificados:")
        direction_counts = Counter(valid_df['direction'])
        for cluster_idx, angle in enumerate(cluster_angles):
            name = cluster_to_name[cluster_idx]
            count = direction_counts.get(name, 0)
            percent = (count / len(valid_df)) * 100 if len(valid_df) > 0 else 0
            print(f"  Grupo {cluster_idx}: {name} (ángulo: {angle:.1f}°, conteo: {count}, {percent:.1f}%)")
        
        # Check for potential overclusterng
        if n_clusters > 3:
            # Look for similar angles that might be the same direction
            for i in range(n_clusters):
                for j in range(i+1, n_clusters):
                    # Check if angles are within 30 degrees of each other
                    angle_diff = min(abs(cluster_angles[i] - cluster_angles[j]), 
                                    360 - abs(cluster_angles[i] - cluster_angles[j]))
                    if angle_diff < 30:
                        print(f"  Nota: Los grupos {i} y {j} tienen ángulos similares " 
                              f"({cluster_angles[i]:.1f}° y {cluster_angles[j]:.1f}°)")
                        print(f"  Estos podrían representar la misma dirección general")
        
        return valid_df, cluster_angles, cluster_to_name
        
    except Exception as e:
        print(f"Error en el análisis de trayectoria: {e}")
        import traceback
        traceback.print_exc()
        return df, None, None
    
def parse_trajectory(trajectory_str):
    """Parse trajectory string into list of coordinate tuples."""
    try:
        # Check if input is a string
        if not isinstance(trajectory_str, str):
            return []
            
        # Extract all (x,y) coordinates from the trajectory string
        coordinates = re.findall(r'\((\d+),(\d+)\)', trajectory_str)
        
        # Convert to list of (x, y) tuples with integer values
        points = [(int(x), int(y)) for x, y in coordinates]
        
        return points
    except Exception as e:
        print(f"Error parsing trajectory: {e} for input: {trajectory_str}")
        return []

def calculate_trajectory_vector(points):
    """Calculate the direction vector of a trajectory."""
    if not points or len(points) < 2:
        return None, None, None
    
    try:
        # Use first and last point for overall direction
        start_x, start_y = points[0]
        end_x, end_y = points[-1]
        
        # Calculate displacement vector
        dx = end_x - start_x
        dy = end_y - start_y
        
        # Calculate vector length (distance traveled)
        distance = math.sqrt(dx**2 + dy**2)
        
        # Calculate angle in degrees (0 = East, going counterclockwise)
        angle = math.degrees(math.atan2(dy, dx))
        
        # Normalize angle to 0-360 range
        if angle < 0:
            angle += 360
            
        return dx, dy, angle
    except Exception as e:
        print(f"Error calculating trajectory vector: {e}")
        return None, None, None

def cluster_directions(df, n_clusters=4):
    """Cluster vehicle trajectories into dominant directions."""
    # Get valid angles
    valid_angles = df[df['angle'] >= 0]['angle'].values
    
    if len(valid_angles) < n_clusters:
        print(f"Advertencia: No hay suficientes trayectorias válidas. Se encontraron {len(valid_angles)}, se necesitan al menos {n_clusters}")
        return None, None
    
    # Convert angles to x,y on unit circle for proper clustering
    x = np.cos(np.radians(valid_angles))
    y = np.sin(np.radians(valid_angles))
    
    # Combine into feature array
    features = np.column_stack((x, y))
    
    # Perform clustering
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    clusters = kmeans.fit_predict(features)
    
    # Get cluster centers
    centers = kmeans.cluster_centers_
    
    # Convert cluster centers back to angles
    cluster_angles = np.degrees(np.arctan2(centers[:, 1], centers[:, 0]))
    cluster_angles = np.mod(cluster_angles, 360)  # Normalize to 0-360
    
    # Function to assign angle to nearest cluster
    def find_nearest_cluster(angle):
        if angle < 0:
            return -1
        
        # Convert angle to point on unit circle
        x = np.cos(np.radians(angle))
        y = np.sin(np.radians(angle))
        
        # Find nearest cluster
        distances = [np.sqrt((x - cx)**2 + (y - cy)**2) for cx, cy in centers]
        return np.argmin(distances)
    
    # Assign each trajectory to a cluster
    df['direction_cluster'] = df['angle'].apply(find_nearest_cluster)
    
    return cluster_angles, df

def name_direction_clusters(cluster_angles, video_id=None):
    """Assign names to direction clusters based on their angles."""
    # Sort clusters by angle
    sorted_indices = np.argsort(cluster_angles)
    sorted_angles = cluster_angles[sorted_indices]
    
    # If video_id is provided, use it as prefix for direction names
    # MODIFICACIÓN: Eliminar el sufijo "_manual" si existe
    if video_id is not None:
        video_id = video_id.replace("_manual", "")
        prefix = f"{video_id}_" 
    else:
        prefix = ""
    
    # Create direction names based on number of clusters
    direction_names = [f"{prefix}Direction_{i+1}" for i in range(len(sorted_angles))]
    
    # Create mapping from cluster index to direction name
    cluster_to_name = {}
    for i, idx in enumerate(sorted_indices):
        cluster_to_name[idx] = direction_names[i]
    
    return cluster_to_name, sorted_angles

def assign_direction_names(df, cluster_to_name):
    """Assign descriptive direction names to trajectories."""
    # Create a mapping function
    def get_direction_name(cluster_idx):
        if cluster_idx < 0:
            return "Unknown"
        return cluster_to_name.get(cluster_idx, f"Direction_{cluster_idx}")
    
    # Apply mapping to create named direction column
    df['direction'] = df['direction_cluster'].apply(get_direction_name)
    
    return df

def visualize_directions(df, cluster_angles, cluster_to_name, file_path, video_path=None):
    """
    Visualiza las trayectorias de vehículos directamente sobre el primer frame del video.
    Asegura que las trayectorias se sobrepongan correctamente a la imagen.
    """
    if cluster_angles is None or cluster_to_name is None:
        return
    
    # Create output filename
    base_filename = file_path.rsplit('.', 1)[0]
    
    # NUEVO: Eliminar "_manual" del título también
    video_id = os.path.splitext(os.path.basename(file_path))[0]
    video_id = video_id.replace("_manual", "")
    
    # Create a colormap for different directions
    direction_to_color = {}
    colors = plt.cm.tab10(np.linspace(0, 1, len(cluster_to_name)))
    for i, cluster_idx in enumerate(cluster_to_name.keys()):
        direction_to_color[cluster_to_name[cluster_idx]] = colors[i]
    
    # Add "Unknown" if present
    if "Unknown" in df['direction'].values:
        direction_to_color["Unknown"] = (0.7, 0.7, 0.7, 1.0)  # Gray
    
    # Extract first frame from video if possible
    first_frame = None
    
    if video_path and os.path.exists(video_path):
        try:
            # Open video and get first frame
            cap = cv2.VideoCapture(video_path)
            if cap.isOpened():
                ret, first_frame = cap.read()
                if not ret:
                    print("Error: No se pudo leer el primer frame del video.")
                    first_frame = None
                cap.release()
            else:
                print("Error: No se pudo abrir el video.")
        except Exception as e:
            print(f"Error al extraer el primer frame: {e}")
    else:
        print("No se proporcionó un video válido para extraer el primer frame.")
    
    # IMPORTANTE: Crear una figura nueva con tamaño específico
    plt.close('all')  # Cerrar figuras existentes para evitar interferencias
    plt.figure(figsize=(14, 10))
    
    # If we have a first frame, use it as background
    if first_frame is not None:
        # Convert BGR to RGB (OpenCV uses BGR, matplotlib uses RGB)
        first_frame_rgb = cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB)
        
        # Obtener dimensiones del frame
        frame_height, frame_width = first_frame_rgb.shape[:2]
        print(f"Dimensiones del frame: {frame_width}x{frame_height}")
        
        # IMPORTANTE: Primero establecer los límites del gráfico
        plt.xlim(0, frame_width)
        plt.ylim(frame_height, 0)  # Invertido para mantener orientación de imagen
        
        # Mostrar la imagen completa (ocupando todo el espacio disponible)
        plt.imshow(first_frame_rgb, extent=[0, frame_width, frame_height, 0])
        print("Usando el primer frame del video como fondo.")
    else:
        # If no frame available, use trajectory data to determine plot boundaries
        all_points = []
        for _, row in df.iterrows():
            all_points.extend(row['trajectory_points'])
        
        if all_points:
            # Determine minimum and maximum coordinates
            min_x = min(p[0] for p in all_points)
            max_x = max(p[0] for p in all_points)
            min_y = min(p[1] for p in all_points)
            max_y = max(p[1] for p in all_points)
            
            # Add some padding
            width = max(max_x - min_x, 100)
            height = max(max_y - min_y, 100)
            frame_width = max_x + width * 0.1
            frame_height = max_y + height * 0.1
        else:
            # Default dimensions if no points
            frame_width, frame_height = 640, 480
        
        plt.gca().set_facecolor('white')
        plt.grid(True, alpha=0.3)
        print("No se pudo obtener el primer frame. Usando fondo blanco.")
        
        # Establecer límites explícitos
        plt.xlim(0, frame_width)
        plt.ylim(frame_height, 0)  # Invertir Y para mantener consistencia
    
    # MODIFICADO: Asegurarnos de que las trayectorias se dibujen completamente dentro del frame
    # Obtenemos las coordenadas máximas para ajustar si es necesario
    all_xs = []
    all_ys = []
    for _, row in df.iterrows():
        points = row['trajectory_points']
        if len(points) >= 2:
            xs, ys = zip(*points)
            all_xs.extend(xs)
            all_ys.extend(ys)
    
    if all_xs and all_ys:
        max_x = max(all_xs)
        max_y = max(all_ys)
        # Si las trayectorias exceden los límites, ajustamos
        if max_x > frame_width or max_y > frame_height:
            # Factor de escala para asegurar que las trayectorias quepan en el frame
            scale_x = frame_width / (max_x + 1) if max_x > frame_width else 1
            scale_y = frame_height / (max_y + 1) if max_y > frame_height else 1
            scale = min(scale_x, scale_y) * 0.9  # Reducir un poco para tener margen
            
            print(f"Aplicando factor de escala {scale} para ajustar trayectorias al frame")
            
            # Necesitaríamos ajustar cada trayectoria antes de dibujar
            for index, row in df.iterrows():
                new_points = [(x * scale, y * scale) for x, y in row['trajectory_points']]
                df.at[index, 'trajectory_points'] = new_points
    
    # Get a sample of trajectories (plot all if few, sample if many)
    sample_size = min(len(df), 500)  # Increased sample size for better representation
    if len(df) > sample_size:
        sample_df = df.sample(sample_size, random_state=42)
    else:
        sample_df = df
    
    # Plot trajectories
    for _, row in sample_df.iterrows():
        points = row['trajectory_points']
        if len(points) >= 2:
            xs, ys = zip(*points)
            color = direction_to_color.get(row['direction'], (0, 0, 0))
            
            # Plot trajectory with higher alpha for better visibility
            plt.plot(xs, ys, '-', alpha=0.7, linewidth=2, color=color)
            
            # Mark start with circle
            plt.plot(xs[0], ys[0], 'o', markersize=5, color=color, alpha=0.8)
            
            # Mark end with arrow
            # Usar una longitud proporcional al tamaño de la imagen para la flecha
            arrow_width = frame_width * 0.01
            arrow_length = frame_height * 0.015
            
            # Solo agregar flecha si hay suficiente espacio
            if len(xs) >= 2:
                plt.arrow(xs[-2], ys[-2], 
                         xs[-1] - xs[-2], ys[-1] - ys[-2],
                         head_width=arrow_width, head_length=arrow_length, 
                         color=color, alpha=0.8)
    
    # Add a legend
    from matplotlib.lines import Line2D
    legend_elements = [Line2D([0], [0], color=color, lw=3, label=direction)
                      for direction, color in direction_to_color.items()]
    
    # Add legend with cluster information - a la derecha para no interferir con la imagen
    legend = plt.legend(handles=legend_elements, loc='upper right', 
                       title="Direcciones Identificadas", framealpha=0.8)
    
    # Add angle information to legend
    for cluster_idx, text in zip(cluster_to_name.keys(), legend.get_texts()):
        direction_name = cluster_to_name[cluster_idx]
        angle = cluster_angles[cluster_idx]
        text.set_text(f"{direction_name} ({angle:.1f}°)")
    
    # Add direction statistics as text annotation
    direction_counts = Counter(df['direction'])
    stats_text = "Estadísticas de Direcciones:\n"
    for direction, count in direction_counts.most_common():
        percent = (count / len(df)) * 100
        stats_text += f"{direction}: {count} ({percent:.1f}%)\n"
    
    # MODIFICADO: Colocar estadísticas en la esquina superior izquierda para no interferir
    plt.text(0.02, 0.07, stats_text, transform=plt.gca().transAxes,
             verticalalignment='bottom', horizontalalignment='left',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # Set title and labels
    plt.title(f'Análisis de Trayectorias de Vehículos - {video_id}')
    
    # Hide axes for cleaner visual
    plt.axis('off')
    
    # Adjust layout to ensure all elements are visible
    plt.tight_layout()
    
    # Save the visualization - sin "_manual" en el nombre del archivo
    output_path = f"{base_filename.replace('_manual', '')}_trajectories.png"
    plt.savefig(output_path, dpi=300, bbox_inches='tight')  # Higher DPI for better quality
    
    print(f"\nVisualización generada: {output_path}")
    return output_path


def analyze_excel_file(excel_path, video_path=None):
    """
    Analiza un archivo Excel con datos de detecciones de vehículos.
    Si se proporciona video_path, utiliza el primer frame para visualizar las trayectorias.
    """
    try:
        # Procesar los datos con agrupamiento adaptativo
        df, cluster_angles, cluster_to_name = process_vehicle_data(
            excel_path, 
            min_clusters=2,  # Mínimo número de grupos a considerar
            max_clusters=10   # Máximo número de grupos a considerar
        )
        
        if df is not None:
            # Guardar los resultados en nuevas hojas del archivo Excel original
            with pd.ExcelWriter(excel_path, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
                # Crear hoja de resumen con conteos de dirección de vehículos
                if cluster_to_name is not None:
                    # Obtener conteos de dirección
                    direction_counts = Counter(df['direction'])
                    
                    # Crear dataframe de resumen
                    summary_df = pd.DataFrame({
                        'Direction': list(direction_counts.keys()),
                        'VehicleCount': list(direction_counts.values()),
                    })
                    
                    # Ordenar por conteo (descendente)
                    summary_df = summary_df.sort_values('VehicleCount', ascending=False)
                    
                    # Agregar porcentajes
                    total = summary_df['VehicleCount'].sum()
                    summary_df['Percentage'] = summary_df['VehicleCount'] / total * 100
                    
                    # Guardar en Excel
                    summary_df.to_excel(writer, sheet_name="Direction_Summary", index=False)
                    
                    # También guardar información de los grupos
                    cluster_info = pd.DataFrame({
                        'Cluster': list(cluster_to_name.keys()),
                        'DirectionName': list(cluster_to_name.values()),
                        'Angle': [cluster_angles[i] for i in cluster_to_name.keys()]
                    })
                    cluster_info.to_excel(writer, sheet_name="Direction_Clusters", index=False)
                
                # Guardar los datos analizados completos
                df.to_excel(writer, sheet_name="Analyzed_Trajectories", index=False)
            
            # Crear visualización de trayectorias sobre el primer frame
            output_path = visualize_directions(df, cluster_angles, cluster_to_name, excel_path, video_path)
            
            # NUEVA ESTRUCTURA: Guardar visualización en la estructura de directorios nueva
            if NUEVA_ESTRUCTURA and output_path and os.path.exists(output_path):
                try:
                    # Identificar ID de cámara y fecha
                    if EXCEL_IDS_PATH and os.path.exists(EXCEL_IDS_PATH):
                        ids_camaras = cargar_ids_desde_excel(EXCEL_IDS_PATH)
                    else:
                        ids_camaras = CAMERA_IDS
                    
                    # Obtener el ID de la cámara y la fecha del nombre del archivo
                    nombre_base = os.path.basename(excel_path)
                    camera_id = identificar_id_camara(nombre_base, ids_camaras)
                    fecha = extraer_fecha_de_nombre(nombre_base)
                    
                    if camera_id:
                        # Guardar en la nueva estructura
                        guardar_archivo_nueva_estructura(
                            output_path,
                            os.path.join(os.getcwd(), "resultados"),
                            camera_id,
                            fecha
                        )
                except Exception as e:
                    print(f"Error al guardar visualización en nueva estructura: {e}")
            
            print("\n¡Análisis completado!")
            print(f"Procesadas {len(df)} trayectorias de vehículos válidas")
            print("\nResultados añadidos como nuevas hojas al archivo Excel original:")
            print("- Analyzed_Trajectories: Análisis detallado de trayectorias")
            print("- Direction_Summary: Conteo de vehículos por dirección")
            print("- Direction_Clusters: Información sobre las agrupaciones de dirección identificadas")
            
            return True
        
    except Exception as e:
        print(f"Error al procesar el archivo: {e}")
        import traceback
        traceback.print_exc()
        
    return False


def procesar_archivos_zip_automatizado(batch_size=5, max_duration_seconds=300):
    """
    Versión automatizada para procesar archivos ZIP con videos en lotes.
    Extrae y procesa batch_size videos a la vez.
    
    Args:
        batch_size (int): Número de videos a procesar por lote
        max_duration_seconds (int): Duración máxima en segundos para procesar un video
    """
    
    try:
        # Definir el directorio de resultados en la carpeta actual del proyecto
        resultados_dir = os.path.join(os.getcwd(), "resultados")
        os.makedirs(resultados_dir, exist_ok=True)
        print(f"Directorio de resultados: {resultados_dir}")

        print("\n" + "="*50)
        print(f"PROCESAMIENTO AUTOMATIZADO DE VIDEOS EN ARCHIVOS ZIP (EN LOTES DE {batch_size})")
        print(f"Omitiendo videos más largos de {max_duration_seconds} segundos ({max_duration_seconds/60:.1f} minutos)")
        print("="*50)
        
        # Inicializar el registro
        registro = RegistroVideos()
        
        # Mostrar estadísticas
        estadisticas = registro.obtener_estadisticas()
        print(f"Registro existente: {estadisticas['total_videos']} videos procesados, "
              f"{estadisticas['completados']} completados, {estadisticas['con_errores']} con errores")
        
        # Buscar archivos ZIP
        print(f"\nBuscando archivos ZIP en: {BASE_DIRECTORY}")
        archivos_zip = encontrar_archivos_zip(BASE_DIRECTORY)
        
        if not archivos_zip:
            print(f"No se encontraron archivos ZIP en {BASE_DIRECTORY}.")
            return
        
        print(f"Se encontraron {len(archivos_zip)} archivos ZIP.")
        
        # Procesar cada archivo ZIP
        for i, ruta_zip in enumerate(archivos_zip):
            nombre_zip = os.path.basename(ruta_zip)
            print(f"\n{'-'*80}")
            print(f"[{i+1}/{len(archivos_zip)}] Procesando archivo: {nombre_zip}")
            
            # Listar los videos - necesitamos duración, así que True en obtener_duracion
            videos_info = listar_videos_en_zip(ruta_zip, obtener_duracion=True)
            videos_procesados = registro.obtener_videos_procesados_por_zip(ruta_zip)
            
            print(f"Videos encontrados: {len(videos_info)}")
            print(f"Videos previamente procesados: {len(videos_procesados)}")
            
            # Filtrar videos para procesar solo los que sean cortos
            videos_filtrados = []
            for video in videos_info:
                # Comprobar duración
                duracion_segundos = video.get("duracion_segundos")
                
                if duracion_segundos is None:
                    print(f"Advertencia: No se pudo determinar la duración de {video['nombre']}. Se excluirá por precaución.")
                    continue
                    
                if duracion_segundos > max_duration_seconds:
                    print(f"Omitiendo {video['nombre']} - Duración: {video['duracion']} (mayor a {max_duration_seconds/60:.1f} minutos)")
                    # Marcar como saltado en el registro
                    registro.marcar_como_procesado(ruta_zip, video['nombre'], "saltado por duración")
                    continue
                    
                videos_filtrados.append(video)
            
            print(f"Videos que cumplen el criterio de duración (<= {max_duration_seconds/60:.1f} minutos): {len(videos_filtrados)}")
            
            # Procesar videos filtrados en lotes
            for j in range(0, len(videos_filtrados), batch_size):
                # Obtener el lote actual de videos
                batch_videos = videos_filtrados[j:j+batch_size]
                print(f"\nProcesando lote {j//batch_size + 1} ({len(batch_videos)} videos)")
                
                # Lista para mantener seguimiento de los directorios temporales
                temp_dirs = []
                temp_paths = []
                
                try:
                    # Primero extraer todos los videos del lote
                    for k, video_info in enumerate(batch_videos):
                        nombre_video = video_info["nombre"]
                        
                        print(f"\n[{j+k+1}/{len(videos_filtrados)}] Extrayendo: {nombre_video}")
                        print(f"Tamaño: {video_info['tamaño']}")
                        print(f"Duración: {video_info['duracion']} ({video_info['duracion_segundos']:.1f} segundos)")
                        
                        # Verificar si el video ya ha sido procesado
                        if not PROCESS_ALL_VIDEOS and registro.esta_procesado(ruta_zip, nombre_video):
                            print("Este video ya ha sido procesado. Saltando...")
                            temp_dirs.append(None)
                            temp_paths.append(None)
                            continue
                        
                        # Extraer el video temporalmente
                        print("Extrayendo video temporalmente...")
                        temp_dir, ruta_temp = extraer_video_temporal(ruta_zip, nombre_video)
                        
                        if not temp_dir or not ruta_temp:
                            print("No se pudo extraer el video temporalmente")
                            temp_dirs.append(None)
                            temp_paths.append(None)
                            continue
                            
                        print(f"Video extraído en: {ruta_temp}")
                        temp_dirs.append(temp_dir)
                        temp_paths.append(ruta_temp)
                    
                    # Ahora procesar cada video extraído
                    for k, video_info in enumerate(batch_videos):
                        if temp_dirs[k] is None:
                            continue  # Saltar videos que no se pudieron extraer
                            
                        nombre_video = video_info["nombre"]
                        ruta_temp = temp_paths[k]
                        
                        print(f"\n[{j+k+1}/{len(videos_filtrados)}] Procesando: {nombre_video}")
                        
                        try:
                            # Procesar el video con YOLOv8
                            print("Procesando video con YOLOv8...")
                            
                            # Obtener nombre base del video (sin extensión ni ruta)
                            nombre_base = os.path.splitext(os.path.basename(nombre_video))[0]
                            
                            # Ajustar el comando YOLOv8 para usar el modelo específico del disco
                            # y evitar descargas automáticas que podrían fallar
                            yolo_model_path = os.path.join(os.getcwd(), "yolov8s.pt")
                            if os.path.exists(yolo_model_path):
                                modelo = yolo_model_path
                            else:
                                # Si no existe, usar un path relativo (asumiendo que está en el directorio actual)
                                modelo = "yolov8s.pt"
                                
                            # CAMBIO IMPORTANTE: Ahora especificamos explícitamente resultados_dir como 
                            # ubicación de salida para el Excel en el script de YOLO
                            # Crear directorio específico para este video
                            video_results_dir = os.path.join(resultados_dir, nombre_base)
                            os.makedirs(video_results_dir, exist_ok=True)
                            
                            # Modificar la variable de entorno PYTHONPATH para garantizar que ultralytics
                            # use nuestro directorio de resultados en lugar del predeterminado
                            env = os.environ.copy()
                            env["PYTHONPATH"] = f"{os.getcwd()}:{env.get('PYTHONPATH', '')}"
                            
                            # Especificar explícitamente la ruta de resultados
                            comando_shell = f'python run_yolo.py model="{modelo}" source="{ruta_temp}" save=True project={video_results_dir}'
                            print(f"Ejecutando comando con shell=True: {comando_shell}")
                            
                            # NUEVO: Guardar la salida del comando para análisis
                            resultado = subprocess.run(comando_shell, shell=True, check=True, 
                                                     stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                                                     text=True, env=env)
                            
                            # Imprimir salida
                            print("SALIDA ESTÁNDAR:")
                            print(resultado.stdout)
                            if resultado.stderr:
                                print("ERRORES:")
                                print(resultado.stderr)
                                
                            # NUEVO: Analizar la salida para extraer el directorio de resultados
                            results_dir = verificar_resultados_yolo(nombre_video, resultado.stdout + resultado.stderr)

                            # Buscar el archivo Excel de resultados
                            excel_path = encontrar_excel_resultado(nombre_video, video_results_dir)
                            
                            # Si no se encuentra el Excel en nuestro directorio, buscar en otros lugares
                            if not excel_path:
                                print("No se encontró el Excel en el directorio especificado, buscando en otras ubicaciones...")
                                excel_path = encontrar_excel_resultado(nombre_video, None)
                            
                            if excel_path and os.path.exists(excel_path):
                                print(f"Archivo de resultados encontrado: {excel_path}")
                                
                                # NUEVO: Si el Excel está en otra ubicación, copiarlo a nuestro directorio de resultados
                                if not excel_path.startswith(resultados_dir):
                                    nuevo_excel_path = os.path.join(video_results_dir, f"{nombre_base}_resultados.xlsx")
                                    try:
                                        shutil.copy2(excel_path, nuevo_excel_path)
                                        print(f"Excel copiado al directorio de resultados: {nuevo_excel_path}")
                                        excel_path = nuevo_excel_path
                                    except Exception as e:
                                        print(f"Error al copiar Excel: {e}")
                                
                                # Marcar como procesado en el registro
                                registro.marcar_como_procesado(ruta_zip, nombre_video, "completado", excel_path)
                                
                                # Analizar resultados si está habilitado
                                if ANALYZE_RESULTS:
                                    print("\nAnalizando resultados de detección...")
                                    success = analyze_excel_file(excel_path, ruta_temp)
                                    
                                    if not success:
                                        print("El análisis de resultados no tuvo éxito.")
                                        registro.marcar_como_procesado(ruta_zip, nombre_video, "completado sin análisis", excel_path)
                            else:
                                print(f"No se encontró el archivo Excel de resultados para {nombre_video}")
                                print("Se requieren resultados reales para el análisis. Continuando con el siguiente video...")
                                registro.marcar_como_procesado(ruta_zip, nombre_video, "error: no se encontró archivo de resultados")
                                
                        except subprocess.CalledProcessError as e:
                            print(f"Error al ejecutar YOLOv8: {e}")
                            if hasattr(e, 'stderr'):
                                print(f"Salida de error: {e.stderr}")
                            if hasattr(e, 'stdout'):
                                print(f"Salida estándar: {e.stdout}")
                            registro.marcar_como_procesado(ruta_zip, nombre_video, "error: falló el procesamiento de YOLO")
                        except Exception as e:
                            print(f"Error al procesar el video: {str(e)}")
                            import traceback
                            traceback.print_exc()
                            registro.marcar_como_procesado(ruta_zip, nombre_video, "error: excepción durante procesamiento")
                            
                finally:
                    # Limpiar archivos temporales después de procesar el lote completo
                    print("\nLimpiando archivos temporales del lote...")
                    for temp_dir in temp_dirs:
                        if temp_dir:
                            try:
                                shutil.rmtree(temp_dir)
                            except Exception as e:
                                print(f"Error al eliminar directorio temporal: {str(e)}")
                    
                    print(f"Lote {j//batch_size + 1} completado")
        
        # Mostrar estadísticas finales
        estadisticas_finales = registro.obtener_estadisticas()
        print("\n" + "="*50)
        print("ESTADÍSTICAS FINALES")
        print("="*50)
        print(f"Total de videos procesados: {estadisticas_finales['total_videos']}")
        print(f"Videos completados: {estadisticas_finales['completados']}")
        print(f"Videos con errores: {estadisticas_finales['con_errores']}")
        print(f"Videos saltados: {estadisticas_finales['saltados']}")
        print(f"Última actualización: {estadisticas_finales['ultima_actualizacion']}")
        print("="*50)
    
    except Exception as e:
        print(f"\nError inesperado: {str(e)}")
        import traceback
        traceback.print_exc()        
        
# Nueva función para buscar archivos Excel recientes en todo el sistema
def find_recent_excel_files(minutes=5):
    """
    Busca archivos Excel modificados en los últimos N minutos.
    
    Args:
        minutes (int): Número de minutos a considerar para la búsqueda
        
    Returns:
        list: Lista de tuplas (ruta_archivo, tiempo_modificacion) ordenada por más recientes
    """
    excel_files = []
    tiempo_actual = time.time()
    tiempo_limite = tiempo_actual - (minutes * 60)  # Convertir minutos a segundos
    
    directorios_busqueda = [
        os.getcwd(),
        os.path.join(os.getcwd(), "runs"),
        os.path.join(os.getcwd(), "runs/detect"),
        os.path.join(os.getcwd(), "resultados"),
        os.path.expanduser("~")
    ]
    
    print(f"Buscando archivos Excel modificados en los últimos {minutes} minutos...")
    
    for directorio in directorios_busqueda:
        if not os.path.exists(directorio):
            continue
            
        print(f"Buscando en: {directorio}")
        
        for ruta, _, archivos in os.walk(directorio):
            for archivo in archivos:
                if archivo.endswith('.xlsx'):
                    ruta_completa = os.path.join(ruta, archivo)
                    try:
                        tiempo_modificacion = os.path.getmtime(ruta_completa)
                        if tiempo_modificacion >= tiempo_limite:
                            excel_files.append((ruta_completa, tiempo_modificacion))
                    except Exception as e:
                        print(f"Error al verificar tiempo de {ruta_completa}: {e}")
    
    # Ordenar por tiempo de modificación (más reciente primero)
    excel_files.sort(key=lambda x: x[1], reverse=True)
    return excel_files

# Add this function to your file - it's referenced but missing in your implementation
def verificar_resultados_yolo(nombre_video, script_output):
    """
    Analiza la salida del script de YOLO para identificar el directorio de resultados
    y buscar pistas sobre la ubicación del Excel.
    
    Args:
        nombre_video (str): Nombre del video procesado
        script_output (str): Salida del script de YOLO
        
    Returns:
        str: Ruta al directorio de resultados, o None si no se encuentra
    """
    # Buscar patrón "Results saved to **runs/detect/X**" en la salida
    pattern = r'Results saved to \*\*(.*?)\*\*'
    match = re.search(pattern, script_output)
    
    if match:
        results_dir = match.group(1)
        print(f"Se identificó el directorio de resultados de YOLO: {results_dir}")
        
        # Verificar si existe
        if os.path.exists(results_dir):
            print(f"El directorio de resultados existe")
            # Buscar archivos Excel en ese directorio
            for archivo in os.listdir(results_dir):
                if archivo.endswith('.xlsx'):
                    ruta_completa = os.path.join(results_dir, archivo)
                    print(f"Encontrado Excel en directorio de resultados: {ruta_completa}")
                    return results_dir
            
            print("No se encontraron archivos Excel en el directorio de resultados")
        else:
            print(f"El directorio de resultados no existe: {results_dir}")
    else:
        print("No se pudo identificar el directorio de resultados en la salida del script")
    
    return None

def main():
    # Define aquí la función que hace la exportación
    def export_function():
        return export_to_excel(output_dir, video_name)
    
    # Define la condición para verificar si hay datos
    def hay_datos():
        return len(vehicle_data) > 0
    
    # Configura los manejadores
    setup_export_handlers(export_function, hay_datos)
    
    # Resto de tu código main()...

def main():
    print("\n" + "="*50)
    print("SISTEMA DE ANÁLISIS DE VIDEOS AUTOMATIZADO")
    print("="*50)
        
    try:
        # Procesar archivos ZIP automáticamente con límite de duración de 2 minutos (120 segundos)
        procesar_archivos_zip_automatizado(batch_size=5, max_duration_seconds=300)
        
        print("\nProcesamiento automático completado.")
    except KeyboardInterrupt:
        # El handler ya maneja la exportación
        print("\nPrograma interrumpido por el usuario.")
    except Exception as e:
        print(f"\nError inesperado: {str(e)}")
        # Exportar en caso de error
        if len(vehicle_data) > 0:
            export_to_excel(output_dir, video_name)
    finally:
        print("\nPrograma finalizado.")


if __name__ == "__main__":
    main()


SISTEMA DE ANÁLISIS DE VIDEOS AUTOMATIZADO
Directorio de resultados: /Users/valeriaserna/versionfinalacaso/resultados

PROCESAMIENTO AUTOMATIZADO DE VIDEOS EN ARCHIVOS ZIP (EN LOTES DE 5)
Omitiendo videos más largos de 300 segundos (5.0 minutos)
Registro existente: 450 videos procesados, 116 completados, 9 con errores

Buscando archivos ZIP en: /Volumes/Elements/Daniela
Se encontraron 80 archivos ZIP.

--------------------------------------------------------------------------------
[1/80] Procesando archivo: Sep 1-2.zip
Videos encontrados: 40
Videos previamente procesados: 40
Omitiendo 20210901/_0_1_093BD6DAC8B2435E9C4499B07B27466A_/20210901_054902.mp4 - Duración: 16:33:26 (mayor a 5.0 minutos)
Omitiendo 20210901/_0_1_0D98647C6C454A96A0E5D5C407C6D4F9_/20210901_054856.mp4 - Duración: 16:33:32 (mayor a 5.0 minutos)
Omitiendo 20210901/_0_1_1B4B1918188F4451B9227A5D7E006223_/20210901_054909.mp4 - Duración: 16:33:20 (mayor a 5.0 minutos)
Omitiendo 20210901/_0_1_36FE410A053F49F1B5B86FB5DDE75

OpenCV: Couldn't read video stream from file "/var/folders/b5/6d3jsq3n12j27vfn8yqh_y500000gn/T/tmp3g783xl3/_0_1_47047A09D7F746059FAF11F6535B03ED_/20210910_052937.mp4"


Videos encontrados: 38
Videos previamente procesados: 38
Omitiendo _0_1_093BD6DAC8B2435E9C4499B07B27466A_/20210910_052944.mp4 - Duración: 00:11:47 (mayor a 5.0 minutos)
Omitiendo _0_1_093BD6DAC8B2435E9C4499B07B27466A_/20210910_054656.mp4 - Duración: 16:13:22 (mayor a 5.0 minutos)
Omitiendo _0_1_0D98647C6C454A96A0E5D5C407C6D4F9_/20210910_052859.mp4 - Duración: 00:12:37 (mayor a 5.0 minutos)
Omitiendo _0_1_0D98647C6C454A96A0E5D5C407C6D4F9_/20210910_054649.mp4 - Duración: 16:13:30 (mayor a 5.0 minutos)
Omitiendo _0_1_1B4B1918188F4451B9227A5D7E006223_/20210910_052907.mp4 - Duración: 00:12:28 (mayor a 5.0 minutos)
Omitiendo _0_1_1B4B1918188F4451B9227A5D7E006223_/20210910_054702.mp4 - Duración: 16:13:15 (mayor a 5.0 minutos)
Omitiendo _0_1_36FE410A053F49F1B5B86FB5DDE75E86_/20210910_054714.mp4 - Duración: 16:13:05 (mayor a 5.0 minutos)
Omitiendo _0_1_3AF47518AAC44BE28059405D87237362_/20210910_054720.mp4 - Duración: 16:12:59 (mayor a 5.0 minutos)
Advertencia: No se pudo determinar la duración 

In [ ]:
# Ruta al Excel generado (ajusta la ruta según donde se guardó tu archivo)
#excel_path = "/Users/valeriaserna/anaconda3/envs/TC2004B/lib/python3.11/site-packages/ultralytics/resultados/20210121_deteccion_vehiculos_20250411_103434.xlsx"

# Ejecuta el análisis manualmente
#analyze_excel_file(excel_path)

In [ ]:
from signal_handlers import setup_export_handlers

ImportError: cannot import name 'setup_export_handlers' from 'signal_handlers' (c:\Users\AxelC\Downloads\bueno\signal_handlers.py)